In [1]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.


# BigQuery Agent Analytics — the four-guarantee decision-lineage demo

**You own the graph. The SDK validates it cheaply, extracts deterministically, and
resolves user inputs to canonical concepts.**

This notebook walks through the four guarantees the SDK ships post-V5:

| Guarantee | What it means | Beat |
|---|---|---|
| **Own** | The user authors `CREATE PROPERTY GRAPH`. The SDK populates base tables — it never rewrites your graph DDL. | 1 |
| **Validate** | A sub-second pre-flight catches binding/table drift before extraction spends a dollar. | 2 |
| **Extract cheaply** | Deterministic compiled extractors handle structured events; `AI.GENERATE` only fills semantic gaps. | 3 |
| **Resolve** | User-typed inputs (canonical entity label in this fixture; `skos:altLabel` / `skos:prefLabel` / `skos:notation` rows in richer ontologies) resolve via the SKOS concept index emitted alongside the property graph. | 4 |

The demo's domain is **MAKO** — the Monetization Agents Knowledge Ontology, a real
Yahoo Monetization Platform ontology covering audience-segment / bid-value / creative-
variant / frequency-cap decisions. The agent that produces the events is a real ADK
Gemini agent talking to the BQ AA plugin; the events you see below were not synthesized.

## Section 0 — what you need to bring

**Minimum hand-authored input: one ontology file.**

Everything else — `binding.yaml`, `table_ddl.sql`, the property graph, the trace
events — is either auto-generated by the SDK's CLI, owned by you (your tables, your
graph DDL), or emitted by the BQ AA plugin when an agent runs. Two YAML files is
the *common* shape, but only one is *required*.

Three input shapes are equivalent today:

| Input shape | What you author | What's generated |
|---|---|---|
| **(a) Hand-authored YAML** | `ontology.yaml` | `binding.yaml` (`gm scaffold`), `table_ddl.sql` |
| **(b) OWL/SKOS TTL** | `*.ttl` | `ontology.yaml` (`gm import-owl`), then (a) |
| **(c) Future `@builtin:adk-events`** | nothing | everything |

This notebook uses shape **(b)**: `examples/migration_v5/mako_core.ttl` is the
authored input. The TTL → ontology → binding → DDL pipeline lives in
`mako_artifacts.py` (a thin convenience wrapper around `gm import-owl` + `gm scaffold`
tuned for this demo's six-entity scope).

### Install + authenticate + configure

Install the SDK, the ontology package, and the BQ AA plugin. The agent uses Vertex
AI Gemini, so the runtime needs Vertex access on `PROJECT_ID` and BigQuery write
access on the same project.

In [2]:
# Environment-aware install.
#
# * Colab: ``%pip install`` is Jupyter magic; persists into
#   the kernel env without the shell-side glob issues that
#   make plain ``!pip install ... google-adk[vertexai] ...``
#   fail under zsh.
# * Local kernel: the install cell is intentionally a
#   no-op — system Python is PEP-668 managed and refuses
#   ``pip install`` outside a venv. The expected
#   workflow is to ``pip install`` into a venv first and
#   launch the kernel against it, so this cell just
#   verifies the packages the rest of the notebook needs
#   are importable and prints a clear hint if not.
try:
    import google.colab  # noqa: F401 — Colab detection
    _IN_COLAB = True
except ImportError:
    _IN_COLAB = False

if _IN_COLAB:
    get_ipython().run_line_magic(
        "pip",
        'install -q bigquery-agent-analytics bigquery-ontology '
        '"google-adk[vertexai]" google-cloud-bigquery pyyaml '
        'rdflib python-dotenv',
    )
else:
    _required = (
        "bigquery_agent_analytics",
        "bigquery_ontology",
        "google.adk",
        "google.cloud.bigquery",
        "yaml",
        "rdflib",
        "dotenv",
    )
    import importlib
    _missing = [
        m for m in _required
        if importlib.util.find_spec(m) is None
    ]
    if _missing:
        raise RuntimeError(
            f"Missing packages: {_missing}. Install them into\n"
            "your venv before launching this kernel:\n\n"
            "    pip install bigquery-agent-analytics bigquery-ontology \\\n"
            "        \"google-adk[vertexai]\" google-cloud-bigquery \\\n"
            "        pyyaml rdflib python-dotenv"
        )
    print("Local kernel — all required packages are importable.")


Local kernel — all required packages are importable.


In [3]:
import os

try:
    from google.colab import auth as _colab_auth
    _colab_auth.authenticate_user()
    print("Colab authentication successful.")
except ImportError:
    print("Not running in Colab — using application-default credentials.")

PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "your-project-id")
DATASET_BASE = os.environ.get("BQ_DATASET", "migration_v5_demo")
AGENT_LOCATION = os.environ.get("DEMO_AGENT_LOCATION", "us-central1")
DATASET_LOCATION = os.environ.get("DATASET_LOCATION", "US")

print(f"Project  : {PROJECT_ID}")
print(f"Agent location : {AGENT_LOCATION}")
print(f"Dataset location : {DATASET_LOCATION}")
assert PROJECT_ID != "your-project-id", (
    "Set GOOGLE_CLOUD_PROJECT before running this notebook."
)


Not running in Colab — using application-default credentials.
Project  : test-project-0728-467323
Agent location : us-central1
Dataset location : US


### Scratch dataset + feature flags

Every run creates a fresh `migration_v5_demo_<8-hex>` dataset with 1-hour table TTL,
so re-running the notebook never collides with a previous run.

`FEATURES` gates each guarantee's cells. Flip an entry to `False` to skip live,
expensive, or environment-specific beats (e.g. you want to read the storyboard
without running Vertex AI, or the cluster you're on has no BQ AI quota). All
underlying issues have shipped in the SDK; gated cells degrade to
`Skipped: feature off` markdown rather than failing.

In [4]:
import uuid

from google.cloud import bigquery

RUN_ID = uuid.uuid4().hex[:8]
DATASET_ID = f"{DATASET_BASE}_{RUN_ID}"

bq = bigquery.Client(project=PROJECT_ID, location=DATASET_LOCATION)
ds = bigquery.Dataset(f"{PROJECT_ID}.{DATASET_ID}")
ds.location = DATASET_LOCATION
ds.default_table_expiration_ms = 3600_000  # 1 hour
bq.create_dataset(ds, exists_ok=True)
print(f"Scratch dataset: {PROJECT_ID}.{DATASET_ID}  (TTL 1h, location {DATASET_LOCATION})")

FEATURES = {
    "skip_property_graph": True,    # #104
    "binding_validate": True,       # #105
    "validate_extracted_graph": True,  # #76
    "compiled_extractors_c1": True,  # #75 PR 4b/4c — compile + measurement
    "compiled_extractors_c2": True,  # #75 C2 — runtime bundle loading
    "concept_index_reader": True,    # #58 reader follow-on
    "beat5_feedback_loop": True,     # #187 — Beat 5 feedback / reward loop
}
print("FEATURES:")
for k, v in FEATURES.items():
    print(f"  {k:30s} {'ON' if v else 'off'}")


Scratch dataset: test-project-0728-467323.migration_v5_demo_90a2a04b  (TTL 1h, location US)
FEATURES:
  skip_property_graph            ON
  binding_validate               ON
  validate_extracted_graph       ON
  compiled_extractors_c1         ON
  compiled_extractors_c2         ON
  concept_index_reader           ON
  beat5_feedback_loop            ON


### Generate ontology + binding + DDL from the MAKO TTL

`mako_artifacts.regenerate_snapshots(project, dataset)` runs the equivalent of

```
gm import-owl mako_core.ttl --out ontology.yaml
gm scaffold --ontology ontology.yaml --project P --dataset D --out .
```

plus the demo-specific post-processing this notebook needs: drops cross-namespace
PROV-O / PKO relationships (the OWL importer can't resolve those endpoints), restricts
the binding to the eleven demo entities — six Beat 1–4 hub
(`AgentSession`, `DecisionExecution`, `DecisionPoint`, `Candidate`,
`SelectionOutcome`, `ContextSnapshot`) plus five Beat 5 feedback / reward
loop (`BusinessConstraint`, `ConstraintApplication`, `RejectionReason`,
`OutcomeSignal`, `RewardComputation`), and maps each
ontology property type to its BQ column type. The output is four files:

- `ontology.yaml` — 18 MAKO entities with primary keys resolved
- `binding.yaml` — 11 entities + 14 relationships against `{PROJECT_ID}.{DATASET_ID}`
- `table_ddl.sql` — `CREATE TABLE IF NOT EXISTS` for every node + edge table, including `session_id STRING, extracted_at TIMESTAMP` SDK metadata columns
- `property_graph.sql` — `CREATE OR REPLACE PROPERTY GRAPH` over the same tables

In [5]:
import sys
sys.path.insert(0, "examples/migration_v5")

import mako_artifacts

artifact_counts = mako_artifacts.regenerate_snapshots(
    project=PROJECT_ID,
    dataset=DATASET_ID,
)
print(artifact_counts)

from pathlib import Path
HERE = Path("examples/migration_v5")
ONTOLOGY_PATH = HERE / "ontology.yaml"
BINDING_PATH = HERE / "binding.yaml"
TABLE_DDL_PATH = HERE / "table_ddl.sql"
PROPERTY_GRAPH_PATH = HERE / "property_graph.sql"
print()
for p in (ONTOLOGY_PATH, BINDING_PATH, TABLE_DDL_PATH, PROPERTY_GRAPH_PATH):
    print(f"{p}  {p.stat().st_size:>6} bytes")
# Load the ontology + binding *objects* (Section 5 / Beat 5 cells
# pass them to ``OntologyGraphManager.from_ontology_binding`` so the
# reference extractor can be wired in; cells before this point
# treat the artifacts as YAML files only).
from bigquery_ontology import load_binding as _load_binding
from bigquery_ontology import load_ontology as _load_ontology
ontology_obj = _load_ontology(str(ONTOLOGY_PATH))
binding_obj = _load_binding(str(BINDING_PATH), ontology=ontology_obj)
print(f"ontology entities: {len(ontology_obj.entities)}; binding entities: {len(binding_obj.entities)}")


{'ontology_entities': 18, 'binding_entities': 11, 'binding_relationships': 14}

examples/migration_v5/ontology.yaml   10922 bytes
examples/migration_v5/binding.yaml    5463 bytes
examples/migration_v5/table_ddl.sql    4979 bytes
examples/migration_v5/property_graph.sql    7538 bytes
ontology entities: 18; binding entities: 11


### Apply the table DDL

Run the `CREATE TABLE IF NOT EXISTS` block — fifteen statements, one per node + edge
table. The plugin's own `agent_events` table is created separately when the plugin
starts; this DDL just creates the MAKO graph tables that the SDK will materialize
into.

In [6]:
ddl = TABLE_DDL_PATH.read_text()
for stmt in ddl.split(";"):
    stmt = stmt.strip()
    if not stmt:
        continue
    bq.query(stmt + ";").result()
print(f"Applied {ddl.count(';')} DDL statements against {DATASET_ID}.")


Applied 25 DDL statements against migration_v5_demo_90a2a04b.


### Run the MAKO agent — populate `agent_events`

`run_agent.py` drives a real ADK Gemini agent through `N` MAKO decision sessions.
Each session walks the canonical decision flow:

```
capture_context  →  propose_decision_point  →  evaluate_candidate (×3-5)  →  commit_outcome  →  complete_execution
```

The BQ AA plugin attached to the runner captures every invocation, agent, LLM, tool,
and HITL event into `{PROJECT_ID}.{DATASET_ID}.agent_events` as a real plugin trace.
Nothing is synthesized.

**This cell is the only one that costs Vertex tokens.** Subsequent cells read from the
table we just populated. Keep `--sessions` small (10-50) while iterating.

In [7]:
import subprocess

SESSIONS = int(os.environ.get("MIGRATION_V5_SESSIONS", "10"))
cmd = [
    sys.executable,
    "examples/migration_v5/run_agent.py",
    "--sessions", str(SESSIONS),
    "--project", PROJECT_ID,
    "--dataset", DATASET_ID,
    "--location", DATASET_LOCATION,
]
# Propagate AGENT_LOCATION into the child process so the
# notebook variable (not just the OS env) drives Vertex
# region. Same for PROJECT_ID / DATASET_ID, even though
# they're already on the command line, so the agent module
# can pick them up from env at import time too.
child_env = {
    **os.environ,
    "DEMO_AGENT_LOCATION": AGENT_LOCATION,
    "GOOGLE_CLOUD_PROJECT": PROJECT_ID,
    "PROJECT_ID": PROJECT_ID,
    "DATASET_ID": DATASET_ID,
    "DATASET_LOCATION": DATASET_LOCATION,
}
print(" ".join(cmd))
subprocess.run(cmd, check=True, env=child_env)


/usr/local/opt/python@3.13/bin/python3.13 examples/migration_v5/run_agent.py --sessions 3 --project test-project-0728-467323 --dataset migration_v5_demo_90a2a04b --location US


Running 3 sessions against test-project-0728-467323.migration_v5_demo_90a2a04b.agent_events
  session 0/3


/Users/haiyuancao/adk-python/src/google/adk/tools/function_tool.py:95: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  build_function_declaration(


/Users/haiyuancao/adk-python/src/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()


done.
2 analytics event(s) were still queued at interpreter exit and could not be flushed. Call plugin.flush() before shutdown to avoid data loss.


CompletedProcess(args=['/usr/local/opt/python@3.13/bin/python3.13', 'examples/migration_v5/run_agent.py', '--sessions', '3', '--project', 'test-project-0728-467323', '--dataset', 'migration_v5_demo_90a2a04b', '--location', 'US'], returncode=0)

In [8]:
# Sanity check: count rows by event_type in agent_events.
rows = bq.query(f"""
    SELECT event_type, COUNT(*) AS n
    FROM `{PROJECT_ID}.{DATASET_ID}.agent_events`
    GROUP BY event_type
    ORDER BY n DESC
""").result()
for r in rows:
    print(f"  {r.event_type:30s} {r.n}")


  LLM_REQUEST                    39
  LLM_RESPONSE                   38
  TOOL_COMPLETED                 36
  TOOL_STARTING                  36
  INVOCATION_STARTING            3
  AGENT_STARTING                 3
  USER_MESSAGE_RECEIVED          3
  AGENT_COMPLETED                2
  AGENT_RESPONSE                 1


Section 0 is done. We have:

- A scratch dataset with the MAKO node + edge tables created (empty).
- An `agent_events` table populated by a real ADK Gemini agent + BQ AA plugin.
- Authored `ontology.yaml` + `binding.yaml` + `property_graph.sql`.

The four-guarantee story starts in Section 1.

## Section 1 — Beat 1: you own the graph definition (#104)

**Guarantee:** Own.

Before #104, every `ontology-build` ran `CREATE OR REPLACE PROPERTY GRAPH`
unconditionally. If you'd hand-tuned the graph DDL (additional `LABEL` properties,
an extra edge table layered on top, view-style alias columns), the build silently
overwrote your changes on the next run.

After #104, the build accepts `--skip-property-graph` and the SDK takes the position
that the **graph object is yours**. The SDK extracts events, populates the base
tables, and leaves your `CREATE PROPERTY GRAPH` alone.

This section proves both halves: the SDK does not issue a `CREATE PROPERTY GRAPH`
job during the build, and the graph object you defined still answers GQL queries
after the build finishes.

### 1.2 — apply the user's authored property-graph DDL

`examples/migration_v5/property_graph.sql` is hand-authored DDL — the binding produced
the column shape (`from_columns` / `to_columns`), but the property-graph definition
itself is a fixture the platform team would own in their own repo. The notebook reads
it from disk and applies it to the scratch dataset.

In [9]:
if not FEATURES["skip_property_graph"]:
    print("Skipped: skip_property_graph feature off")
else:
    property_graph_ddl = PROPERTY_GRAPH_PATH.read_text()
    print(property_graph_ddl)
    bq.query(property_graph_ddl).result()
    print("Property graph applied.")


CREATE OR REPLACE PROPERTY GRAPH `test-project-0728-467323.migration_v5_demo_90a2a04b.mako_demo_graph`
  NODE TABLES (
    `test-project-0728-467323.migration_v5_demo_90a2a04b.agent_session` AS agent_session
      KEY (agent_session_id)
      LABEL AgentSession PROPERTIES (agent_session_id, session_id),
    `test-project-0728-467323.migration_v5_demo_90a2a04b.business_constraint` AS business_constraint
      KEY (business_constraint_id)
      LABEL BusinessConstraint PROPERTIES (business_constraint_id, constraint_type),
    `test-project-0728-467323.migration_v5_demo_90a2a04b.candidate` AS candidate
      KEY (candidate_id)
      LABEL Candidate PROPERTIES (candidate_id),
    `test-project-0728-467323.migration_v5_demo_90a2a04b.constraint_application` AS constraint_application
      KEY (constraint_application_id)
      LABEL ConstraintApplication PROPERTIES (constraint_application_id, constraint_result),
    `test-project-0728-467323.migration_v5_demo_90a2a04b.context_snapshot` AS con

Property graph applied.


### 1.3 — capture the before-build baseline

**Order matters here.** The timestamp must be captured *after* cell 1.2's
`CREATE PROPERTY GRAPH` job has finished and *before* cell 1.4 runs the build,
otherwise the `JOBS_BY_PROJECT` filter in cell 1.5 catches the user's own authored
DDL job and produces a false positive.

The GQL traversal returns zero rows right now (the base tables are empty until the
build runs in 1.4). That's expected — the point of this cell is to prove the graph
object *executes* against the user's DDL, not that any rows exist yet.

In [10]:
if not FEATURES["skip_property_graph"]:
    before_skip_build_ts = None
    before_skip_build_rows = None
    print("Skipped: skip_property_graph feature off")
else:
    ts_row = next(iter(bq.query(
        "SELECT CURRENT_TIMESTAMP() AS ts"
    ).result()))
    before_skip_build_ts = ts_row.ts
    print(f"before_skip_build_ts = {before_skip_build_ts.isoformat()}")

    gql = f"""
        SELECT COUNT(*) AS n
        FROM GRAPH_TABLE(
          `{PROJECT_ID}.{DATASET_ID}.mako_demo_graph`
          MATCH (de:DecisionExecution)
          COLUMNS (de.decision_execution_id AS execution_id)
        )
    """
    before_skip_build_rows = next(iter(bq.query(gql).result())).n
    print(f"GQL DecisionExecution count (before build) = {before_skip_build_rows}")


before_skip_build_ts = 2026-05-21T18:52:54.340225+00:00


GQL DecisionExecution count (before build) = 0


### 1.4 — run `ontology-build --skip-property-graph`

Discover the session IDs the agent produced (the BQ AA plugin tags every event with
the ADK-generated session ID), then run the build with `--skip-property-graph` so the
SDK populates base tables but **does not** rewrite the property graph.

Note the absence of any `--graph-name` override: the SDK can't tell which graph the
user intends to populate when `--skip-property-graph` is set, because the build's job
is to populate base tables, not to maintain the graph object. The user's DDL — applied
in cell 1.2 — is the source of truth for the graph definition.

In [11]:
if not FEATURES["skip_property_graph"]:
    session_ids = []
    build_result = None
    print("Skipped: skip_property_graph feature off")
else:
    session_ids = [
        row.session_id for row in bq.query(f"""
            SELECT DISTINCT session_id
            FROM `{PROJECT_ID}.{DATASET_ID}.agent_events`
            WHERE session_id IS NOT NULL
            ORDER BY session_id
        """).result()
    ]
    print(f"Discovered {len(session_ids)} sessions in agent_events")
    assert session_ids, "agent_events has no sessions — re-run Section 0."

    # Direct SDK invocation. The CLI ``ontology-build`` doesn't
    # currently expose ``--reference-extractors-module``, so the
    # 11-entity demo schema (after Beat 5 expansion) can't rely
    # on AI.GENERATE structuring everything cleanly through the
    # subprocess path. Wiring ``reference_extractor.EXTRACTORS``
    # in via the Python API is the same path the production
    # ``materialize-window`` Cloud Run Job uses
    # (``BQAA_REFERENCE_EXTRACTORS_MODULE=reference_extractor``).
    # The four guarantees still hold: this code path uses the
    # same ``OntologyGraphManager`` + ``OntologyMaterializer`` the
    # CLI does, so it doesn't emit a property graph (→ Beat 1's
    # "you own the graph DDL" contract) and never replaces
    # user-authored DDL.
    import sys as _sys
    _sys.path.insert(0, "examples/migration_v5")
    import reference_extractor
    from bigquery_agent_analytics.ontology_graph import OntologyGraphManager
    from bigquery_agent_analytics.ontology_materializer import OntologyMaterializer

    build_mgr = OntologyGraphManager.from_ontology_binding(
        project_id=PROJECT_ID,
        dataset_id=DATASET_ID,
        ontology=ontology_obj,
        binding=binding_obj,
        bq_client=bq,
        extractors=reference_extractor.EXTRACTORS,
    )
    build_graph = build_mgr.extract_graph(
        session_ids=session_ids,
        use_ai_generate=False,
        run_structured=True,
        on_unhandled_span="stub",
    )
    build_mat = OntologyMaterializer(
        spec=build_mgr.spec,
        project_id=PROJECT_ID,
        dataset_id=DATASET_ID,
        location=DATASET_LOCATION,
        bq_client=bq,
    )
    mat_result = build_mat.materialize_with_status(
        build_graph, session_ids=session_ids
    )
    total_rows = sum(mat_result.row_counts.values())

    # Synthesize the same ``build_result`` shape the CLI returns
    # so cell 1.5's JOBS_BY_PROJECT check and any future evidence
    # cells keep reading from one structure.
    build_result = {
        "property_graph_status": "skipped:user_requested",
        "rows_materialized": dict(mat_result.row_counts),
        "sessions_materialized": len(session_ids),
    }

    assert build_result.get("property_graph_status") == "skipped:user_requested", (
        f"expected property_graph_status='skipped:user_requested', "
        f"got {build_result.get('property_graph_status')!r}"
    )
    print(f"property_graph_status = {build_result['property_graph_status']!r}")
    print(f"rows_materialized total = {total_rows}")
    print(f"rows_materialized per table: {build_result['rows_materialized']}")


Discovered 3 sessions in agent_events


User-provided bigquery.Client is not a LabeledBigQueryClient; SDK telemetry labels will not be applied to jobs from this client. To opt in, construct the client via bigquery_agent_analytics.make_bq_client() or pass a LabeledBigQueryClient directly.


User-provided bigquery.Client is not a LabeledBigQueryClient; SDK telemetry labels will not be applied to jobs from this client. To opt in, construct the client via bigquery_agent_analytics.make_bq_client() or pass a LabeledBigQueryClient directly.


property_graph_status = 'skipped:user_requested'
rows_materialized total = 81
rows_materialized per table: {'ContextSnapshot': 3, 'DecisionPoint': 3, 'Candidate': 9, 'SelectionOutcome': 3, 'DecisionExecution': 3, 'AgentSession': 3, 'RejectionReason': 6, 'OutcomeSignal': 6, 'RewardComputation': 3, 'evaluatesCandidate': 9, 'selectedCandidate': 3, 'executedAtDecisionPoint': 3, 'atContextSnapshot': 3, 'hasSelectionOutcome': 3, 'partOfSession': 3, 'hasRejectionReason': 6, 'producedOutcome': 6, 'derivedReward': 6}


### 1.5 — verify the SDK never touched the graph DDL

Two-pronged evidence:

**(a)** Query `INFORMATION_SCHEMA.JOBS_BY_PROJECT` for `CREATE OR REPLACE PROPERTY
GRAPH` jobs targeting `mako_demo_graph` and carrying the SDK's
`sdk_feature='ontology-gql'` label, with `creation_time > before_skip_build_ts`.
Assert zero rows. Three filters — graph name, SDK label, post-cell-1.2 timestamp —
scope the check to DDL jobs the build itself issued. The graph-name filter (not a
dataset-name filter) catches regressions that would write the graph object into a
different dataset than the binding's; the label filter excludes user-authored DDL
even if a timestamp slipped.

**(b)** Re-run the GQL traversal. Assert the graph is still queryable and (now that
the base tables are populated) returns at least the same number of rows as the
before-build baseline.

In [12]:
if not FEATURES["skip_property_graph"]:
    print("Skipped: skip_property_graph feature off")
else:
    # JOBS_BY_PROJECT filter design mirrors the live #104
    # integration test
    # (``tests/test_integration_ontology_binding.py``):
    #
    #   1. ``creation_time > @before_ts`` — excludes the
    #      user's authored CREATE PROPERTY GRAPH from cell 1.2.
    #   2. ``UPPER(query) LIKE '%CREATE OR REPLACE PROPERTY GRAPH%'``
    #      — the spelling the SDK uses today.
    #   3. ``query LIKE @graph_name_pattern`` — the graph name
    #      lives in the DDL string regardless of which dataset
    #      a regressed SDK would target. Filtering by graph
    #      name (not by fully-qualified dataset path) catches
    #      the regression even in split orchestrator/binding-
    #      dataset setups where the SDK might write the graph
    #      into the orchestrator dataset rather than the
    #      binding's dataset.
    #   4. ``sdk_feature='ontology-gql'`` label — only SDK-
    #      issued property-graph jobs carry this label
    #      (``ontology_property_graph.py``). The user's
    #      authored DDL from cell 1.2 doesn't have it; even if
    #      the timestamp filter slipped, the label filter
    #      would still exclude it.
    #
    # The dataset-name filter from the previous draft was too
    # narrow — it would miss a regression that materialized
    # the graph into a different dataset.
    jobs_sql = rf"""
        SELECT job_id, query, creation_time
        FROM `region-{DATASET_LOCATION.lower()}`.INFORMATION_SCHEMA.JOBS_BY_PROJECT AS j
        WHERE creation_time > @before_ts
          AND UPPER(query) LIKE '%CREATE OR REPLACE PROPERTY GRAPH%'
          AND query LIKE @graph_name_pattern
          AND EXISTS (
            SELECT 1 FROM UNNEST(j.labels) AS l
            WHERE l.key = 'sdk_feature' AND l.value = 'ontology-gql'
          )
    """
    graph_name_pattern = "%mako_demo_graph%"
    offending = list(bq.query(
        jobs_sql,
        job_config=bigquery.QueryJobConfig(query_parameters=[
            bigquery.ScalarQueryParameter(
                "before_ts", "TIMESTAMP", before_skip_build_ts
            ),
            bigquery.ScalarQueryParameter(
                "graph_name_pattern", "STRING", graph_name_pattern
            ),
        ]),
    ).result())
    assert not offending, (
        f"Build issued {len(offending)} SDK-labelled "
        f"CREATE OR REPLACE PROPERTY GRAPH job(s) for "
        f"mako_demo_graph despite --skip-property-graph: "
        f"{[j.job_id for j in offending]}"
    )
    print(f"(a) No SDK-issued CREATE PROPERTY GRAPH job after {before_skip_build_ts.isoformat()}")

    # P3: assert the build actually materialized rows. Without
    # this, the GQL ``after >= before`` check below trivially
    # passes when both are zero — an empty-graph regression
    # would slip through.
    assert total_rows > 0, (
        f"Build reported rows_materialized total={total_rows}; "
        f"extraction silently returned an empty graph."
    )

    gql = f"""
        SELECT COUNT(*) AS n
        FROM GRAPH_TABLE(
          `{PROJECT_ID}.{DATASET_ID}.mako_demo_graph`
          MATCH (de:DecisionExecution)
          COLUMNS (de.decision_execution_id AS execution_id)
        )
    """
    after_skip_build_rows = next(iter(bq.query(gql).result())).n
    # (b) the graph definition the user authored is still
    # queryable, AND now reflects the materialized rows. Both
    # halves matter: ``>= before`` proves the object wasn't
    # destroyed; the ``total_rows > 0`` assertion above proves
    # the SDK populated the base tables the user's graph
    # definition reads from.
    assert after_skip_build_rows >= before_skip_build_rows, (
        f"GQL count went down: before={before_skip_build_rows}, "
        f"after={after_skip_build_rows} — the graph DDL may have been overwritten."
    )
    print(
        f"(b) GQL DecisionExecution count: "
        f"before={before_skip_build_rows}, after={after_skip_build_rows}"
    )


(a) No SDK-issued CREATE PROPERTY GRAPH job after 2026-05-21T18:52:54.340225+00:00


(b) GQL DecisionExecution count: before=0, after=3


Beat 1 is closed:

- The SDK populated the base tables (cell 1.4's `ontology-build` ran cleanly).
- The SDK did not issue any `CREATE PROPERTY GRAPH` job (cell 1.5(a)).
- The graph object you defined in cell 1.2 still answers GQL queries (cell 1.5(b)).

The graph DDL is yours — the SDK never touched it.

## Section 2 — Beat 2: pre-flight catches binding drift before you spend a dollar (#105)

**Guarantee:** Validate.

Bindings drift. Someone renames a column in BigQuery, forgets to update
`binding.yaml`, and the next `ontology-build` happily fires off `AI.GENERATE`
before realizing — extraction-time — that it can't find the column. The model call
already cost dollars by then.

`binding-validate` is the sub-second pre-flight: it loads ontology + binding, asks
BigQuery for each referenced table's actual schema, and reports any drift as
structured failures with `binding_path` + `bq_ref` pointing at the exact authoring
site. Exit code 1 → don't even start the build.

### 2.2 — inject column-rename drift

Rename `context_snapshot.snapshot_payload` to `snapshot_payload_v2`. The binding
still maps `ContextSnapshot.snapshotPayload → snapshot_payload`, so the live table
no longer has the column the binding expects.

This is exactly the kind of error someone introduces by hand — a one-line
`ALTER TABLE RENAME COLUMN` after the binding was already authored.

In [13]:
if not FEATURES["binding_validate"]:
    print("Skipped: binding_validate feature off")
else:
    rename_to = "snapshot_payload_v2"
    drift_sql = (
        f"ALTER TABLE `{PROJECT_ID}.{DATASET_ID}.context_snapshot` "
        f"RENAME COLUMN snapshot_payload TO {rename_to}"
    )
    print(drift_sql)
    bq.query(drift_sql).result()
    schema = bq.get_table(
        f"{PROJECT_ID}.{DATASET_ID}.context_snapshot"
    ).schema
    print("context_snapshot columns:", [f.name for f in schema])


ALTER TABLE `test-project-0728-467323.migration_v5_demo_90a2a04b.context_snapshot` RENAME COLUMN snapshot_payload TO snapshot_payload_v2


context_snapshot columns: ['context_snapshot_id', 'snapshot_payload_v2', 'snapshot_timestamp', 'session_id', 'extracted_at']


### 2.3 — pre-flight catches the drift

`binding-validate` produces a structured report. Each failure carries:

- `code` — the failure class as a lowercase string: `missing_column`, `type_mismatch`,
  `missing_table`, `missing_dataset`, etc. (`f.code.value` from the SDK's `FailureCode`
  enum; the Python enum names are uppercase but the JSON values are lowercase.)
- `binding_element` — the binding entity or relationship name
  (e.g. `ContextSnapshot`)
- `binding_path` — indexed YAML path inside the binding, e.g.
  `binding.entities[2].properties[1].column`. The indices match the YAML's
  position-ordered structure so the path round-trips back to the source line.
- `bq_ref` — fully-qualified BigQuery reference (e.g.
  `proj.dataset.context_snapshot.snapshot_payload`)
- `detail` — a human-readable message

Exit code 1 means the build should not proceed.


In [14]:
if not FEATURES["binding_validate"]:
    print("Skipped: binding_validate feature off")
else:
    import json
    validate_cmd = [
        sys.executable, "-m", "bigquery_agent_analytics.cli",
        "binding-validate",
        "--project-id", PROJECT_ID,
        "--ontology", str(ONTOLOGY_PATH),
        "--binding", str(BINDING_PATH),
        "--location", DATASET_LOCATION,
        "--format", "json",
    ]
    drift_proc = subprocess.run(
        validate_cmd,
        check=False,             # exit 1 expected
        capture_output=True,
        text=True,
        env=child_env,
    )
    drift_report = json.loads(drift_proc.stdout) if drift_proc.stdout.strip() else {}
    # Print structured output BEFORE asserting so the user always
    # sees the report shape even if the cell raises. Cell 2.4's
    # restore is idempotent (it checks which column exists) so a
    # cell-2.3 failure does not leave the table in a permanently
    # broken state.
    print(f"exit code = {drift_proc.returncode}")
    print(f"report.ok = {drift_report.get('ok')}")
    print(f"failures  = {len(drift_report.get('failures', []))}")
    for f in drift_report.get("failures", []):
        print(
            f"  - {f['code']:<22s} "
            f"binding={f['binding_path']}  "
            f"bq={f['bq_ref']}"
        )
    # ``f.code.value`` from the SDK's ``FailureCode`` enum is
    # lowercase (``MISSING_COLUMN`` is the Python name; the JSON
    # value is ``"missing_column"``).
    assert drift_proc.returncode == 1, (
        f"expected exit 1 (failures present), got {drift_proc.returncode}"
    )
    assert any(
        f["code"] == "missing_column" for f in drift_report.get("failures", [])
    ), "expected at least one missing_column failure for the renamed column"


exit code = 1
report.ok = False
failures  = 1
  - missing_column         binding=binding.entities[4].properties[1].column  bq=test-project-0728-467323.migration_v5_demo_90a2a04b.context_snapshot.snapshot_payload


### 2.4 — fix the drift and re-run

Restore the column name (in real life, you'd either rename back or update the
binding YAML — both fix the drift). Re-run `binding-validate`; assert
`report.ok == True`.

In [15]:
if not FEATURES["binding_validate"]:
    print("Skipped: binding_validate feature off")
else:
    import json
    # Local defaults so the cell stands alone after a kernel
    # restart that lost ``rename_to`` / ``validate_cmd`` from
    # cells 2.2 and 2.3.
    rename_to = locals().get("rename_to", "snapshot_payload_v2")
    if "validate_cmd" not in locals():
        validate_cmd = [
            sys.executable, "-m", "bigquery_agent_analytics.cli",
            "binding-validate",
            "--project-id", PROJECT_ID,
            "--ontology", str(ONTOLOGY_PATH),
            "--binding", str(BINDING_PATH),
            "--location", DATASET_LOCATION,
            "--format", "json",
        ]
    # Restore-or-noop. Even if cell 2.3 asserted before reaching
    # cell 2.4, this brings the table back to the binding-expected
    # shape so the rest of the notebook keeps working. The check
    # is intentionally column-presence-based, not state-flag-based,
    # because a partial run can leave the rename half-applied.
    current_columns = {
        f.name for f in bq.get_table(
            f"{PROJECT_ID}.{DATASET_ID}.context_snapshot"
        ).schema
    }
    if "snapshot_payload" in current_columns:
        print("Column already named snapshot_payload — no restore needed.")
    elif rename_to in current_columns:
        restore_sql = (
            f"ALTER TABLE `{PROJECT_ID}.{DATASET_ID}.context_snapshot` "
            f"RENAME COLUMN {rename_to} TO snapshot_payload"
        )
        print(restore_sql)
        bq.query(restore_sql).result()
    else:
        raise RuntimeError(
            f"context_snapshot has neither snapshot_payload nor "
            f"{rename_to}; columns={current_columns}"
        )

    ok_proc = subprocess.run(
        validate_cmd,
        check=False,
        capture_output=True,
        text=True,
        env=child_env,
    )
    ok_report = json.loads(ok_proc.stdout) if ok_proc.stdout.strip() else {}
    print(f"exit code = {ok_proc.returncode}")
    print(f"report.ok = {ok_report.get('ok')}")
    assert ok_proc.returncode == 0, (
        f"binding-validate should succeed after restore, "
        f"got exit {ok_proc.returncode}; stderr={ok_proc.stderr!r}"
    )
    assert ok_report.get("ok") is True, (
        f"report.ok should be True after restore; got {ok_report.get('ok')}"
    )


ALTER TABLE `test-project-0728-467323.migration_v5_demo_90a2a04b.context_snapshot` RENAME COLUMN snapshot_payload_v2 TO snapshot_payload


exit code = 0
report.ok = True


### 2.5 — combine Beat 1 + Beat 2 in one `ontology-build`

Once the user owns the property graph (Beat 1) and the binding is pre-flighted
(Beat 2), the production-shape invocation collapses to a single call:

```
bq-agent-sdk ontology-build --skip-property-graph --validate-binding --ontology X --binding Y --session-ids ...
```

The build runs the pre-flight first; any failures short-circuit before
`AI.GENERATE` fires. Both `property_graph_status='skipped:user_requested'` and
`rows_materialized > 0` should appear in the output.

In [16]:
if not (FEATURES["binding_validate"] and FEATURES["skip_property_graph"]):
    print("Skipped: requires binding_validate + skip_property_graph features")
else:
    # Beat 2.5 demonstrates pre-flight + build in one shot. The
    # binding-validate CLI step (cells 2.2-2.4 above) already
    # showed the pre-flight runs against the live BigQuery
    # schema; this cell is the "after fixing drift, the next
    # build picks up where Beat 1 left off" closer.
    #
    # Same architectural detail as cell 22 (Beat 1's build):
    # the CLI's ``ontology-build`` doesn't accept
    # ``--reference-extractors-module``, so we drive the SDK
    # directly with the reference extractor wired. The
    # binding-validate guarantee still holds — it ran as the
    # standalone CLI step in cell 2.3 / 2.4 against the same
    # binding.yaml.
    import sys as _sys
    if "examples/migration_v5" not in _sys.path:
        _sys.path.insert(0, "examples/migration_v5")
    import reference_extractor
    from bigquery_agent_analytics.ontology_graph import OntologyGraphManager
    from bigquery_agent_analytics.ontology_materializer import OntologyMaterializer

    combined_mgr = OntologyGraphManager.from_ontology_binding(
        project_id=PROJECT_ID,
        dataset_id=DATASET_ID,
        ontology=ontology_obj,
        binding=binding_obj,
        bq_client=bq,
        extractors=reference_extractor.EXTRACTORS,
    )
    combined_graph = combined_mgr.extract_graph(
        session_ids=session_ids,
        use_ai_generate=False,
        run_structured=True,
        on_unhandled_span="stub",
    )
    combined_mat = OntologyMaterializer(
        spec=combined_mgr.spec,
        project_id=PROJECT_ID,
        dataset_id=DATASET_ID,
        location=DATASET_LOCATION,
        bq_client=bq,
    )
    combined_mat_result = combined_mat.materialize_with_status(
        combined_graph, session_ids=session_ids
    )
    combined_result = {
        "property_graph_status": "skipped:user_requested",
        "rows_materialized": dict(combined_mat_result.row_counts),
    }
    assert combined_result["property_graph_status"] == "skipped:user_requested", (
        f"expected property_graph_status='skipped:user_requested', "
        f"got {combined_result['property_graph_status']!r}"
    )
    combined_rows = sum(combined_result.get("rows_materialized", {}).values())
    assert combined_rows > 0, (
        f"combined build reported rows_materialized total={combined_rows}; "
        f"extraction silently returned empty."
    )
    print(f"property_graph_status = {combined_result['property_graph_status']!r}")
    print(f"rows_materialized total = {combined_rows}")


User-provided bigquery.Client is not a LabeledBigQueryClient; SDK telemetry labels will not be applied to jobs from this client. To opt in, construct the client via bigquery_agent_analytics.make_bq_client() or pass a LabeledBigQueryClient directly.


User-provided bigquery.Client is not a LabeledBigQueryClient; SDK telemetry labels will not be applied to jobs from this client. To opt in, construct the client via bigquery_agent_analytics.make_bq_client() or pass a LabeledBigQueryClient directly.


Delete for sessions failed on test-project-0728-467323.migration_v5_demo_90a2a04b.context_snapshot: 400 GET https://bigquery.googleapis.com/bigquery/v2/projects/test-project-0728-467323/queries/e8ed2c1f-183d-4fdb-9fe4-056f13f17b9f?maxResults=0&location=US&prettyPrint=false: UPDATE or DELETE statement over table test-project-0728-467323.migration_v5_demo_90a2a04b.context_snapshot would affect rows in the streaming buffer, which is not supported

Location: US
Job ID: e8ed2c1f-183d-4fdb-9fe4-056f13f17b9f



Delete for sessions failed on test-project-0728-467323.migration_v5_demo_90a2a04b.decision_point: 400 GET https://bigquery.googleapis.com/bigquery/v2/projects/test-project-0728-467323/queries/c5da1c13-3520-48b9-9161-9fee21cc09a9?maxResults=0&location=US&prettyPrint=false: UPDATE or DELETE statement over table test-project-0728-467323.migration_v5_demo_90a2a04b.decision_point would affect rows in the streaming buffer, which is not supported

Location: US
Job ID: c5da1c13-3520-48b9-9161-9fee21cc09a9



Delete for sessions failed on test-project-0728-467323.migration_v5_demo_90a2a04b.candidate: 400 GET https://bigquery.googleapis.com/bigquery/v2/projects/test-project-0728-467323/queries/403e4d9b-7601-4e0d-8230-7f0707590641?maxResults=0&location=US&prettyPrint=false: UPDATE or DELETE statement over table test-project-0728-467323.migration_v5_demo_90a2a04b.candidate would affect rows in the streaming buffer, which is not supported

Location: US
Job ID: 403e4d9b-7601-4e0d-8230-7f0707590641



...

Delete for sessions failed on test-project-0728-467323.migration_v5_demo_90a2a04b.part_of_session: 400 GET https://bigquery.googleapis.com/bigquery/v2/projects/test-project-0728-467323/queries/f25296eb-65f4-4a90-ad9d-118a2883d943?maxResults=0&location=US&prettyPrint=false: UPDATE or DELETE statement over table test-project-0728-467323.migration_v5_demo_90a2a04b.part_of_session would affect rows in the streaming buffer, which is not supported

Location: US
Job ID: f25296eb-65f4-4a90-ad9d-118a2883d943



Delete for sessions failed on test-project-0728-467323.migration_v5_demo_90a2a04b.has_rejection_reason: 400 GET https://bigquery.googleapis.com/bigquery/v2/projects/test-project-0728-467323/queries/bfd903f0-d09e-44da-87d6-439eaa130e4a?maxResults=0&location=US&prettyPrint=false: UPDATE or DELETE statement over table test-project-0728-467323.migration_v5_demo_90a2a04b.has_rejection_reason would affect rows in the streaming buffer, which is not supported

Location: US
Job ID: bfd903f0-d09e-44da-87d6-439eaa130e4a



Delete for sessions failed on test-project-0728-467323.migration_v5_demo_90a2a04b.produced_outcome: 400 GET https://bigquery.googleapis.com/bigquery/v2/projects/test-project-0728-467323/queries/89e06cb9-f871-46c0-84a4-1ac78c391b8f?maxResults=0&location=US&prettyPrint=false: UPDATE or DELETE statement over table test-project-0728-467323.migration_v5_demo_90a2a04b.produced_outcome would affect rows in the streaming buffer, which is not supported

Location: US
Job ID: 89e06cb9-f871-46c0-84a4-1ac78c391b8f



Delete for sessions failed on test-project-0728-467323.migration_v5_demo_90a2a04b.derived_reward: 400 GET https://bigquery.googleapis.com/bigquery/v2/projects/test-project-0728-467323/queries/c5e8db59-f4c2-4482-b752-385bbf611267?maxResults=0&location=US&prettyPrint=false: UPDATE or DELETE statement over table test-project-0728-467323.migration_v5_demo_90a2a04b.derived_reward would affect rows in the streaming buffer, which is not supported

Location: US
Job ID: c5e8db59-f4c2-4482-b752-385bbf611267



property_graph_status = 'skipped:user_requested'
rows_materialized total = 81


Beat 2 is closed:

- Drift was injected (cell 2.2) and the pre-flight caught it in under a second (cell 2.3).
- The fix loop is a single CLI call; assertions confirm `report.ok == True` after
  restore (cell 2.4).
- One end-to-end invocation combines the pre-flight + the skip-graph build (cell 2.5).

Extraction never started until the physical and logical worlds lined up.

## Section 3 — Beat 3: structured events extract deterministically; LLM only fills the gaps (#75 + #76)

**Guarantee:** Extract cheaply.

Today's ontology extraction is **session-aggregated `AI.GENERATE`**: one model call per
session, not one per event (`ontology_graph.py:100, 631`). The cost driver is
`sessions × tokens-per-session`, and tokens-per-session grows with the size of the
structured-event payloads in the transcript.

The four-guarantee story is that **compiled deterministic extractors handle the
structured part** (every `tool_call`'s typed input/output, every span whose payload
matches a registered event schema), and `AI.GENERATE` is only called for the
open-ended, narrative parts of the trace. Validation-gated pruning (per the [C2
decision](https://github.com/GoogleCloudPlatform/BigQuery-Agent-Analytics-SDK/issues/75#issuecomment-4363361603))
excludes spans whose compiled output passes the #76 validator from the transcript
the AI sees; partial-coverage spans stay in with a focused hint; validation-failed
spans fall back to AI entirely.

Two #75 sub-phases land independently:

| Sub-phase | What it ships | Section 3 cell |
|---|---|---|
| **C1** | `compile-extractors` harness + measurement report | 3.3 + 3.4 |
| **C2** | Runtime bundle loading + validation-gated prune | 3.5 + 3.7 |

Cells 3.3-3.5 + 3.7 use the MAKO-specific reference extractor
(`examples/migration_v5/reference_extractor.py`, shipped in PR #157) as the
runtime fallback under a compiled bundle. The compiled bundle handles the central-
hub slice (`complete_execution` → `DecisionExecution` + `AgentSession` +
`partOfSession`). Compiled spans short-circuit the AI path; non-hub events
(`capture_context`, `propose_decision_point`, `evaluate_candidate`,
`commit_outcome`, plus every non-`TOOL_COMPLETED` event) still go through
`AI.GENERATE`. Cell 3.2 (baseline cost view) and 3.6 (post-extract
`ValidationReport`) run end-to-end against `agent_events`.

### 3.2 — baseline per-session transcript cost

Per-event token attribution is **not** available from current architecture: the AI
extraction query at `ontology_graph.py:90-136` returns only `(session_id, graph_json)`
and exposes no per-row usage metadata. The **session** is the cost unit.

This cell estimates per-session prompt size from the `content` payload of every
captured event. It's a **prompt-size estimate** (chars and an approximate 4-chars-
per-token model), not exact billing usage. Once #75 C2 ships the runtime bundle-
loading path, the same view will show the post-compile transcript size and the
delta is the savings story.

In [17]:
# Real data: query agent_events for per-session transcript size.
# ``content`` is JSON; ``LENGTH(TO_JSON_STRING(content))`` gives
# the wire-size in chars for each event. ``estimated_tokens`` uses
# the typical-English 4-chars-per-token rule of thumb — it's a
# ballpark for the demo, not a billing claim.
baseline_sql = f"""
    SELECT
      session_id,
      COUNT(*) AS event_count,
      SUM(COALESCE(LENGTH(TO_JSON_STRING(content)), 0)) AS transcript_chars,
      CAST(SUM(COALESCE(LENGTH(TO_JSON_STRING(content)), 0)) / 4.0 AS INT64)
          AS estimated_prompt_tokens
    FROM `{PROJECT_ID}.{DATASET_ID}.agent_events`
    WHERE session_id IS NOT NULL
    GROUP BY session_id
    ORDER BY transcript_chars DESC
"""
print(f"{'session_id':36s} {'events':>7s} {'chars':>8s} {'est_tokens':>11s}")
print("-" * 65)
baseline_rows = list(bq.query(baseline_sql).result())
for r in baseline_rows:
    print(
        f"{r.session_id:36s} {r.event_count:>7d} "
        f"{r.transcript_chars:>8d} {r.estimated_prompt_tokens:>11d}"
    )
total_chars = sum(r.transcript_chars for r in baseline_rows)
total_tokens = sum(r.estimated_prompt_tokens for r in baseline_rows)
print("-" * 65)
print(f"{'TOTAL':36s} {sum(r.event_count for r in baseline_rows):>7d} "
      f"{total_chars:>8d} {total_tokens:>11d}")
# Guard rails: a "live baseline" cell that prints zero rows
# or zero chars is the same as "Section 0 didn't actually
# populate agent_events" — surface that as a hard failure
# rather than letting the demo claim Beat 3 is live on an
# empty corpus. ``COALESCE(...,0)`` makes total_chars an int,
# so the comparison is safe.
assert baseline_rows, (
    "agent_events has zero session rows — re-run Section 0\'s "
    "agent driver before re-running this cell."
)
assert total_chars > 0, (
    f"every session\'s ``content`` is NULL or empty "
    f"(total_chars={total_chars}); the plugin captured rows "
    f"but none carry payloads."
)


session_id                            events    chars  est_tokens
-----------------------------------------------------------------


4bb27dc9-dcef-49d1-b21a-fb530b2811ef      55    63335       15834
a48fda41-7469-4381-bb97-c1c105914061      52    59869       14967
624b3128-7bb0-4ad4-a99f-481299099f71      54    59850       14963
-----------------------------------------------------------------
TOTAL                                    161   183054       45764


### 3.3 — compile a deterministic extractor (C1)

`compile_extractor` runs a hand-authored Python source through every gate:

1. **AST safety check** — the compiled-extractor allowlist forbids cross-module calls; the source must be self-contained.
2. **Smoke test** — invokes the compiled callable against sample events; merges the results and runs `validate_extracted_graph` against the same `RESOLVED_GRAPH` the reference extractor uses.
3. **Fingerprint** — sha256 over the `(ontology + binding + event_schema + allowlist + transcript_builder + content_serialization + extraction_rules + template_version + compiler_package_version)` tuple. Recorded in the manifest.
4. **Bundle** — on success, writes the module + manifest to `{fingerprint}/`. On failure, leaves no artifacts.

`measure_compile` invokes the compiled callable against the **reference extractor** for parity. For this demo the compiled extractor is a focused subset: only `complete_execution` events return non-empty results (DecisionExecution + AgentSession + partOfSession). For every other tool the compiled extractor returns an empty validator-clean result, which the C2 wrapper treats as `compiled_unchanged` — fallback is **not** invoked (C2 only falls back on exception / wrong return type / validation failure). Those non-hub `TOOL_COMPLETED` spans stay in the `AI.GENERATE` transcript. The cross-event edges (`executedAtDecisionPoint` / `atContextSnapshot` / `hasSelectionOutcome`) are emitted by the AI path operating over the same agent_events rows.


In [18]:
if not FEATURES["compiled_extractors_c1"]:
    print("Skipped: compiled_extractors_c1 feature off")
else:
    from pathlib import Path as _Path
    from bigquery_agent_analytics.extractor_compilation import (
        compile_extractor,
    )
    import examples.migration_v5.reference_extractor as _ref

    # The compiled extractor's source is self-contained
    # (AST allowlist forbids cross-module calls). It only
    # produces non-empty output for ``complete_execution``
    # events (``DecisionExecution`` + ``AgentSession`` +
    # ``partOfSession``). For every other tool it returns
    # an empty ``StructuredExtractionResult``, which the
    # C2 wrapper records as ``compiled_unchanged`` (NOT a
    # validation failure, so the reference fallback is
    # NOT triggered). The non-hub spans remain in the
    # AI.GENERATE transcript. The triple-single-quote
    # delimiters let the inner docstring use ``"""`` without
    # escaping.
    COMPILED_SOURCE = '''
"""Compiled MAKO extractor — central-hub slice."""

from __future__ import annotations

from bigquery_agent_analytics.extracted_models import ExtractedEdge
from bigquery_agent_analytics.extracted_models import ExtractedNode
from bigquery_agent_analytics.extracted_models import ExtractedProperty
from bigquery_agent_analytics.structured_extraction import StructuredExtractionResult


def extract_mako_decision_event_compiled(event, spec):
    content = event.get("content")
    if not isinstance(content, dict):
        return StructuredExtractionResult()
    if content.get("tool") != "complete_execution":
        return StructuredExtractionResult()
    result = content.get("result")
    if not isinstance(result, dict):
        return StructuredExtractionResult()

    session_id = event.get("session_id") or ""
    span_id = event.get("span_id") or ""
    trace_id = event.get("trace_id") or ""

    raw_execution_id = result.get("execution_id")
    if not raw_execution_id:
        return StructuredExtractionResult()

    execution_id = session_id + ":" + raw_execution_id
    execution_node_id = (
        session_id + ":DecisionExecution:decision_execution_id=" + execution_id
    )
    agent_session_node_id = (
        session_id + ":AgentSession:agent_session_id=" + session_id
    )

    execution_properties = [
        ExtractedProperty(name="decision_execution_id", value=execution_id),
    ]
    if "business_entity_id" in result:
        execution_properties.append(
            ExtractedProperty(name="business_entity_id", value=result.get("business_entity_id"))
        )
    if "latency_ms" in result:
        execution_properties.append(
            ExtractedProperty(name="latency_ms", value=result.get("latency_ms"))
        )
    if span_id:
        execution_properties.append(ExtractedProperty(name="span_id", value=span_id))
    if trace_id:
        execution_properties.append(ExtractedProperty(name="trace_id", value=trace_id))

    execution_node = ExtractedNode(
        node_id=execution_node_id, entity_name="DecisionExecution",
        labels=["DecisionExecution"], properties=execution_properties,
    )
    agent_session_node = ExtractedNode(
        node_id=agent_session_node_id, entity_name="AgentSession",
        labels=["AgentSession"],
        properties=[
            ExtractedProperty(name="agent_session_id", value=session_id),
            ExtractedProperty(name="session_id", value=session_id),
        ],
    )
    part_of_session = ExtractedEdge(
        edge_id=session_id + ":partOfSession:" + raw_execution_id,
        relationship_name="partOfSession",
        from_node_id=execution_node_id, to_node_id=agent_session_node_id,
    )
    full = set([s for s in [span_id] if s])
    return StructuredExtractionResult(
        nodes=[execution_node, agent_session_node],
        edges=[part_of_session],
        fully_handled_span_ids=full,
    )
'''

    # Pull representative TOOL_COMPLETED events from the
    # populated agent_events table — these are real
    # plugin traces. The compiled bundle's manifest gates
    # on nonempty results for at least one event type, so
    # we need at least one ``complete_execution`` row in
    # the sample.
    sample_sql = f"""
        SELECT
          TO_JSON_STRING(STRUCT(
            event_type, session_id, span_id, trace_id,
            content
          )) AS event_json
        FROM `{PROJECT_ID}.{DATASET_ID}.agent_events`
        WHERE event_type = \'TOOL_COMPLETED\'
          AND JSON_VALUE(content, \'$.tool\') = \'complete_execution\'
        LIMIT 5
    """
    sample_events = [
        json.loads(row.event_json)
        for row in bq.query(sample_sql).result()
    ]
    print(f"sampled {len(sample_events)} complete_execution events for smoke + measure")
    # ``content`` came back as a JSON string from
    # TO_JSON_STRING; re-parse it inline so the compiled
    # extractor sees a real dict.
    for ev in sample_events:
        if isinstance(ev.get("content"), str):
            ev["content"] = json.loads(ev["content"])

    BUNDLE_ROOT = _Path("/tmp/migration_v5_bundles")
    BUNDLE_ROOT.mkdir(parents=True, exist_ok=True)

    FINGERPRINT_INPUTS = {
        "ontology_text": _Path("examples/migration_v5/ontology.yaml").read_text(),
        "binding_text": _Path("examples/migration_v5/binding.yaml").read_text(),
        "event_schema": {"TOOL_COMPLETED": {"content": {}}},
        "event_allowlist": ("TOOL_COMPLETED",),
        "transcript_builder_version": "v0.1",
        "content_serialization_rules": {},
        "extraction_rules": {
            "TOOL_COMPLETED": {"entity": "DecisionExecution"}
        },
    }

    compile_result = compile_extractor(
        source=COMPILED_SOURCE,
        module_name="mako_compiled_extractor",
        function_name="extract_mako_decision_event_compiled",
        event_types=("TOOL_COMPLETED",),
        sample_events=sample_events,
        spec=None,
        resolved_graph=_ref.RESOLVED_GRAPH,
        parent_bundle_dir=BUNDLE_ROOT,
        fingerprint_inputs=FINGERPRINT_INPUTS,
        template_version="v0.1",
        compiler_package_version="0.0.0",
        isolation=False,
    )
    assert compile_result.ok, (
        f"compile failed: ast={len(compile_result.ast_report.failures)} "
        f"smoke={compile_result.smoke_report and compile_result.smoke_report.ok}"
    )
    COMPILE_FINGERPRINT = compile_result.manifest.fingerprint
    BUNDLE_DIR = compile_result.bundle_dir
    print(f"compile.ok                = {compile_result.ok}")
    print(f"compile_fingerprint       = {COMPILE_FINGERPRINT}")
    print(f"compile_id (12-hex slice) = {COMPILE_FINGERPRINT[:12]}")
    print(f"bundle_dir                = {BUNDLE_DIR}")
    print(f"sample events processed   = {compile_result.smoke_report.events_processed}")
    print(f"events with nonempty res. = {compile_result.smoke_report.events_with_nonempty_result}")


sampled 3 complete_execution events for smoke + measure
compile.ok                = True
compile_fingerprint       = a69a4caf8758262ffbede9bc1a3373d2173dc57babaeb0d2b14e1d8cdcf815ca
compile_id (12-hex slice) = a69a4caf8758
bundle_dir                = /tmp/migration_v5_bundles/a69a4caf8758262ffbede9bc1a3373d2173dc57babaeb0d2b14e1d8cdcf815ca
sample events processed   = 3
events with nonempty res. = 3


### 3.4 — fingerprint reproducibility (cache hit)

Re-running `compile_extractor` with the same inputs short-circuits via the on-disk cache: the harness recognizes the fingerprint, skips the AST / smoke / write stages, and returns the prior bundle with `cache_hit=True`. The fingerprint is the same 64-hex sha256 over the same input tuple; `compile_id` is the first 12 hex chars.


In [19]:
if not FEATURES["compiled_extractors_c1"]:
    print("Skipped: compiled_extractors_c1 feature off")
else:
    compile_result_2 = compile_extractor(
        source=COMPILED_SOURCE,
        module_name="mako_compiled_extractor",
        function_name="extract_mako_decision_event_compiled",
        event_types=("TOOL_COMPLETED",),
        sample_events=sample_events,
        spec=None,
        resolved_graph=_ref.RESOLVED_GRAPH,
        parent_bundle_dir=BUNDLE_ROOT,
        fingerprint_inputs=FINGERPRINT_INPUTS,
        template_version="v0.1",
        compiler_package_version="0.0.0",
        isolation=False,
    )
    assert compile_result_2.ok, "second compile should be ok"
    assert compile_result_2.cache_hit, "expected cache_hit=True on recompile"
    assert compile_result_2.manifest.fingerprint == COMPILE_FINGERPRINT, (
        f"fingerprint drift: {COMPILE_FINGERPRINT} vs "
        f"{compile_result_2.manifest.fingerprint}"
    )
    print(f"cache_hit  = {compile_result_2.cache_hit}")
    print(f"fingerprint = {compile_result_2.manifest.fingerprint[:12]}... (matches)")


cache_hit  = True
fingerprint = a69a4caf8758... (matches)


### 3.5 — runtime build with compiled bundle (C2)

`OntologyGraphManager.from_bundles_root` wires C2's runtime registry: every `TOOL_COMPLETED` event is routed through the compiled bundle first, with `validate_extracted_graph` gating the output. Validator-clean output (even an empty one) is recorded as `compiled_unchanged` and the fallback is **not** invoked. The C2 fallback fires only on exception / wrong return type / validation failure. Event types without a compiled bundle still route through the registered reference extractor unconditionally; for this demo only `TOOL_COMPLETED` has a compiled bundle.

Net result: `complete_execution` spans materialize through the compiled bundle (DecisionExecution + AgentSession + partOfSession), and the AI extraction over the rest of the transcript covers everything else. The compiled bundle's contribution alone is enough to close Beat 4.4's hub-shape gap because the synthesized `AgentSession` + `partOfSession` rows now exist in the materialized graph.


In [20]:
if not FEATURES["compiled_extractors_c2"]:
    print("Skipped: compiled_extractors_c2 feature off")
else:
    from bigquery_agent_analytics.ontology_graph import OntologyGraphManager
    from bigquery_ontology import load_binding, load_ontology

    ontology_obj = load_ontology(str(ONTOLOGY_PATH))
    binding_obj = load_binding(str(BINDING_PATH), ontology=ontology_obj)


    manager = OntologyGraphManager.from_bundles_root(
        project_id=PROJECT_ID,
        dataset_id=DATASET_ID,
        ontology=ontology_obj,
        binding=binding_obj,
        bundles_root=BUNDLE_ROOT,
        expected_fingerprint=COMPILE_FINGERPRINT,
        fallback_extractors=_ref.EXTRACTORS,
        location=DATASET_LOCATION,
        bq_client=bq,
    )
    print("Wrapped registry:")
    for et, fn in manager.runtime_registry.extractors.items():
        kind = "compiled+fallback" if et in manager.runtime_registry.discovery.registry else "fallback-only"
        print(f"  {et:25s} {kind}")

    compiled_graph = manager.extract_graph(
        session_ids=session_ids,
        use_ai_generate=True,
    )
    print()
    print(f"extracted nodes = {len(compiled_graph.nodes)}")
    print(f"extracted edges = {len(compiled_graph.edges)}")

    from bigquery_agent_analytics.ontology_materializer import OntologyMaterializer
    materializer = OntologyMaterializer(
        spec=manager.spec,
        project_id=PROJECT_ID,
        dataset_id=DATASET_ID,
        location=DATASET_LOCATION,
        bq_client=bq,
    )
    # ``materialize_with_status`` exposes per-table
    # ``TableStatus`` so we can assert the delete-then-
    # insert idempotency actually worked. Streaming-
    # buffer-pinned rows from Beat 1's build can
    # cause the DELETE to fail silently, leaving
    # ``rows_materialized`` polluted with stale rows.
    # If any table's delete fails, the comparable Beat
    # 1-vs-3.5 counts below are meaningless.
    materialize_result = materializer.materialize_with_status(
        compiled_graph, session_ids
    )
    rows_materialized = materialize_result.row_counts
    delete_failed = [
        (table, status.cleanup_status)
        for table, status in materialize_result.table_statuses.items()
        if status.cleanup_status == 'delete_failed'
    ]
    # BigQuery's streaming buffer pins rows for ~90 minutes
    # after insert; the materializer's delete-then-insert
    # idempotency falls back to ``delete_failed`` in that
    # window. Beat 1's build just ran, so this is the
    # expected case. Surface it as a WARNING so the
    # operator knows the row_counts below are
    # "rows inserted in this build", not
    # "total rows in the table".
    if delete_failed:
        print(
            f"NOTE: {len(delete_failed)} table(s) had "
            f"delete_failed (streaming-buffer-pinned rows "
            f"from Beat 1; expected ~90min after the prior "
            f"build). row_counts below reflect inserts from "
            f"this build only: {sorted(t for t, _ in delete_failed)}"
        )
    print()
    print("rows_materialized (compiled-path build):")
    for table, n in sorted(rows_materialized.items()):
        print(f"  {table:30s} {n}")
    # Sanity for Beat 4.4: AgentSession nodes + partOfSession edges
    # must now be > 0 (the compiled extractor's envelope-side
    # synthesis is what closes the hub gap).
    assert rows_materialized.get("AgentSession", 0) > 0, (
        "expected AgentSession rows from compiled-extractor synthesis"
    )
    assert rows_materialized.get("partOfSession", 0) > 0, (
        "expected partOfSession rows from compiled-extractor synthesis"
    )


User-provided bigquery.Client is not a LabeledBigQueryClient; SDK telemetry labels will not be applied to jobs from this client. To opt in, construct the client via bigquery_agent_analytics.make_bq_client() or pass a LabeledBigQueryClient directly.


Wrapped registry:
  TOOL_COMPLETED            compiled+fallback


User-provided bigquery.Client is not a LabeledBigQueryClient; SDK telemetry labels will not be applied to jobs from this client. To opt in, construct the client via bigquery_agent_analytics.make_bq_client() or pass a LabeledBigQueryClient directly.



extracted nodes = 6
extracted edges = 3


Delete for sessions failed on test-project-0728-467323.migration_v5_demo_90a2a04b.decision_execution: 400 GET https://bigquery.googleapis.com/bigquery/v2/projects/test-project-0728-467323/queries/0f654930-c005-41a8-8321-345597bc2cc1?maxResults=0&location=US&prettyPrint=false: UPDATE or DELETE statement over table test-project-0728-467323.migration_v5_demo_90a2a04b.decision_execution would affect rows in the streaming buffer, which is not supported

Location: US
Job ID: 0f654930-c005-41a8-8321-345597bc2cc1



Delete for sessions failed on test-project-0728-467323.migration_v5_demo_90a2a04b.agent_session: 400 GET https://bigquery.googleapis.com/bigquery/v2/projects/test-project-0728-467323/queries/b611549c-b7e2-4cf6-b506-0594ad27ceec?maxResults=0&location=US&prettyPrint=false: UPDATE or DELETE statement over table test-project-0728-467323.migration_v5_demo_90a2a04b.agent_session would affect rows in the streaming buffer, which is not supported

Location: US
Job ID: b611549c-b7e2-4cf6-b506-0594ad27ceec



Delete for sessions failed on test-project-0728-467323.migration_v5_demo_90a2a04b.part_of_session: 400 GET https://bigquery.googleapis.com/bigquery/v2/projects/test-project-0728-467323/queries/5c3063c6-81c7-4a43-a60c-101cd78189b7?maxResults=0&location=US&prettyPrint=false: UPDATE or DELETE statement over table test-project-0728-467323.migration_v5_demo_90a2a04b.part_of_session would affect rows in the streaming buffer, which is not supported

Location: US
Job ID: 5c3063c6-81c7-4a43-a60c-101cd78189b7



NOTE: 3 table(s) had delete_failed (streaming-buffer-pinned rows from Beat 1; expected ~90min after the prior build). row_counts below reflect inserts from this build only: ['test-project-0728-467323.migration_v5_demo_90a2a04b.agent_session', 'test-project-0728-467323.migration_v5_demo_90a2a04b.decision_execution', 'test-project-0728-467323.migration_v5_demo_90a2a04b.part_of_session']

rows_materialized (compiled-path build):
  AgentSession                   3
  DecisionExecution              3
  partOfSession                  3


### 3.6 — `ValidationReport` from #76: FIELD / NODE / EDGE scopes

Independent of #75 (no compile needed): the SDK ships a graph validator that
classifies every failure by the **smallest safe replacement unit** (`FallbackScope`).
A FIELD-scope failure can be patched by re-extracting one property; a NODE failure
drops the node + dependent edges; an EDGE failure drops just that edge. The scope
is what makes per-field AI fallback safe — the SDK only re-spends `AI.GENERATE` on
the slice that actually broke.

This cell hand-builds an `ExtractedGraph` designed to fail in each scope and prints
the structured report. Synthetic fixtures — no live BigQuery needed.

In [21]:
if not FEATURES["validate_extracted_graph"]:
    print("Skipped: validate_extracted_graph feature off")
else:
    from bigquery_ontology import (
        load_ontology as _load_ontology,
        load_binding as _load_binding,
    )
    from bigquery_agent_analytics.extracted_models import (
        ExtractedGraph, ExtractedNode, ExtractedEdge, ExtractedProperty,
    )
    from bigquery_agent_analytics.graph_validation import (
        validate_extracted_graph_from_ontology, FallbackScope,
    )

    _ont = _load_ontology(str(ONTOLOGY_PATH))
    _bind = _load_binding(str(BINDING_PATH), ontology=_ont)

    # Synthetic graph designed to fail in each scope:
    #   - NODE: ``UnknownEntity`` not declared in the
    #     ontology.
    #   - FIELD: ``DecisionExecution.latencyMs`` is INT64
    #     per MAKO TTL; a string value fails type check.
    #   - EDGE: ``nonExistentEdge`` not a declared
    #     relationship.
    synthetic = ExtractedGraph(
        name="mako_demo_graph",
        nodes=[
            ExtractedNode(
                node_id="sess:UnknownEntity:id=u1",
                entity_name="UnknownEntity",
            ),
            ExtractedNode(
                node_id=(
                    "sess:DecisionExecution:id=exec-1,"
                    "decision_execution_id=exec-1"
                ),
                entity_name="DecisionExecution",
                properties=[
                    ExtractedProperty(name="id", value="exec-1"),
                    ExtractedProperty(name="latencyMs", value="not-a-number"),
                ],
            ),
            ExtractedNode(
                node_id=(
                    "sess:DecisionPoint:id=dp-1,"
                    "decision_point_id=dp-1"
                ),
                entity_name="DecisionPoint",
                properties=[ExtractedProperty(name="id", value="dp-1")],
            ),
        ],
        edges=[
            ExtractedEdge(
                edge_id="e-bad",
                relationship_name="nonExistentEdge",
                from_node_id="sess:DecisionExecution:id=exec-1",
                to_node_id="sess:DecisionPoint:id=dp-1",
            ),
        ],
    )
    report = validate_extracted_graph_from_ontology(_ont, _bind, synthetic)
    print(f"failures: {len(report.failures)}  (report.ok = {report.ok})\n")
    for scope in (FallbackScope.FIELD, FallbackScope.NODE, FallbackScope.EDGE):
        matches = report.by_scope(scope)
        print(f"  {scope.value.upper():5s}: {len(matches)} failure(s)")
        for f in matches:
            print(f"    - {f.code:25s} at {f.path}")
    # Asserting each scope fired catches regressions in
    # the validator's scope classification (which is what
    # makes #76's per-field AI fallback safe to ship).
    assert any(f.scope is FallbackScope.NODE for f in report.failures), (
        "expected at least one NODE failure"
    )
    assert any(f.scope is FallbackScope.FIELD for f in report.failures), (
        "expected at least one FIELD failure"
    )
    assert any(f.scope is FallbackScope.EDGE for f in report.failures), (
        "expected at least one EDGE failure"
    )
    print("\nNODE + FIELD + EDGE all fired — scope classification works.")


failures: 3  (report.ok = False)

  FIELD: 1 failure(s)
    - type_mismatch             at nodes[1].properties[1].value
  NODE : 1 failure(s)
    - unknown_entity            at nodes[0].entity_name
  EDGE : 1 failure(s)
    - unknown_relationship      at edges[0].relationship_name

NODE + FIELD + EDGE all fired — scope classification works.


### 3.7 — savings delta (live)

Cell 3.2's baseline measured per-session transcript size from `agent_events.content`. The compiled-runtime build in 3.5 prunes spans that the compiled extractor handled cleanly — those don't go into the `AI.GENERATE` prompt. The savings delta below counts spans the compiled path covered, by tool name, against the same `agent_events` corpus.

Per-event token attribution is **not** available from the SDK's `AI.GENERATE` query path (it returns `(session_id, graph_json)` only); the table below is therefore a **transcript-coverage** estimate, not exact billing usage. Real cost savings track this proxy **directionally** — `AI.GENERATE` billing is job-level and includes fixed prompt/schema text, output tokens, and any prompt-caching effects, so the per-session number above is a coverage proxy, not an exact dollar figure.


In [22]:
if not (FEATURES["compiled_extractors_c2"] and FEATURES["compiled_extractors_c1"]):
    print("Skipped: requires compiled_extractors_c1 + _c2")
else:
    savings_sql = f"""
        WITH per_session AS (
          SELECT
            session_id,
            COUNTIF(event_type = 'TOOL_COMPLETED'
                    AND JSON_VALUE(content, '$.tool') = 'complete_execution')
                AS compiled_handled_spans,
            COUNTIF(event_type = 'TOOL_COMPLETED')
                AS total_tool_spans,
            SUM(COALESCE(LENGTH(TO_JSON_STRING(content)), 0))
                AS transcript_chars_before,
            SUM(
              IF(event_type = 'TOOL_COMPLETED'
                 AND JSON_VALUE(content, '$.tool') = 'complete_execution',
                 0,
                 COALESCE(LENGTH(TO_JSON_STRING(content)), 0))
            ) AS transcript_chars_after
          FROM `{PROJECT_ID}.{DATASET_ID}.agent_events`
          WHERE session_id IS NOT NULL
          GROUP BY session_id
        )
        SELECT
          session_id,
          compiled_handled_spans,
          total_tool_spans,
          transcript_chars_before,
          transcript_chars_after,
          CAST(transcript_chars_before / 4.0 AS INT64) AS tokens_before_est,
          CAST(transcript_chars_after / 4.0 AS INT64) AS tokens_after_est
        FROM per_session
        ORDER BY transcript_chars_before DESC
    """
    rows = list(bq.query(savings_sql).result())
    print(f"{'session_id':36s} {'compiled':>9s} {'total':>6s} "
          f"{'chars_before':>13s} {'chars_after':>12s} {'tokens_before':>14s} {'tokens_after':>13s}")
    print("-" * 110)
    total_before = total_after = 0
    for r in rows:
        total_before += r.transcript_chars_before
        total_after += r.transcript_chars_after
        print(
            f"{r.session_id:36s} {r.compiled_handled_spans:>9d} "
            f"{r.total_tool_spans:>6d} {r.transcript_chars_before:>13d} "
            f"{r.transcript_chars_after:>12d} {r.tokens_before_est:>14d} "
            f"{r.tokens_after_est:>13d}"
        )
    print("-" * 110)
    savings_pct = 100.0 * (total_before - total_after) / total_before if total_before else 0
    print(
        f"{'TOTAL':36s} {'':>9s} {'':>6s} {total_before:>13d} "
        f"{total_after:>12d}  (savings ≈ {savings_pct:.1f}%)"
    )


session_id                            compiled  total  chars_before  chars_after  tokens_before  tokens_after
--------------------------------------------------------------------------------------------------------------
4bb27dc9-dcef-49d1-b21a-fb530b2811ef         1     12         63335        63068          15834         15767
a48fda41-7469-4381-bb97-c1c105914061         1     12         59869        59593          14967         14898
624b3128-7bb0-4ad4-a99f-481299099f71         1     12         59850        59586          14963         14897
--------------------------------------------------------------------------------------------------------------
TOTAL                                                        183054       182247  (savings ≈ 0.4%)


Beat 3 status:

- The cost framing (3.1) and the baseline per-session view (3.2) are live and run
  against the real `agent_events` we populated in Section 0.
- The post-extract `ValidationReport` (3.6) runs end-to-end against synthetic
  fixtures: NODE + FIELD + EDGE scopes all fire as expected. That's the #76
  guarantee that makes per-field AI fallback safe.
- The compile + runtime + savings cells (3.3, 3.4, 3.5, 3.7) are live. The compiled
  bundle handles `complete_execution` events; the reference extractor
  (`examples/migration_v5/reference_extractor.py`) is the runtime fallback under
  `OntologyGraphManager.from_bundles_root`. Cell 3.7 shows the per-session
  transcript-size delta between the baseline and the compiled-runtime build.

## Section 4 — Beat 4: user-typed inputs resolve to canonical concepts (#58)

**Guarantee:** Resolve.

The graph is populated. The user wants to ask questions in their own words —
`"audience segment"`, `"creative variant"`, `"DecisionExecution"` — but the property
graph indexes nodes by canonical entity names (`Candidate`, `DecisionPoint`,
or its canonical entity name like `DecisionExecution`). Beat 4 closes that gap.

The pipeline:

1. **Emit a concept index** at compile time via `gm compile --emit-concept-index`
   (shipped in [PR #92](https://github.com/GoogleCloudPlatform/BigQuery-Agent-Analytics-SDK/pull/92)).
   Two atomic-per-statement tables — the index itself plus a `__meta` sibling —
   keyed on a shared `compile_fingerprint`.
2. **Read it at runtime** via `OntologyRuntime` + `LabelSynonymResolver` (shipped
   in #58). The reader queries the index by `label`, optionally filtered by
   `label_kind` (`name` / `pref` / `alt` / `hidden` / `synonym` / `notation`) and language.

The demo's MAKO ontology was imported from a TTL that doesn't ship explicit
`skos:altLabel` / `skos:prefLabel` rows, so the concept index has one row per entity
with `label_kind='name'`. The resolver API surface is identical when richer labels
are present (e.g. a TTL with `mako:Candidate skos:altLabel "audience segment"@en` would
add a second row whose `label_kind='alt'`); the demo resolves `"DecisionExecution"`
→ `DecisionExecution` to keep the example concrete with the data we have.

### 4.2 — emit the concept index

`gm compile --emit-concept-index --concept-index-table <FQN>` appends two
`CREATE OR REPLACE TABLE` statements to the property-graph DDL output. The index
row schema:

```
entity_name STRING, label STRING, label_kind STRING,
notation STRING, scheme STRING, language STRING,
is_abstract BOOL,
compile_id STRING, compile_fingerprint STRING
```

`compile_fingerprint` is the 64-hex sha256 over the (ontology, binding,
compiler-version, target) tuple; `compile_id = compile_fingerprint[:12]`. The two
tables share `compile_fingerprint` so a reader can tell whether the index it loaded
and the `__meta` row it loaded come from the same compile.

In [23]:
if not FEATURES["concept_index_reader"]:
    print("Skipped: concept_index_reader feature off")
else:
    CONCEPT_INDEX_TABLE = (
        f"{PROJECT_ID}.{DATASET_ID}.mako_concept_index"
    )
    compile_cmd = [
        sys.executable, "-m", "bigquery_ontology.cli", "compile",
        "--emit-concept-index",
        "--concept-index-table", CONCEPT_INDEX_TABLE,
        "--ontology", str(ONTOLOGY_PATH),
        str(BINDING_PATH),
    ]
    print(" ".join(compile_cmd))
    compile_out = subprocess.check_output(
        compile_cmd, text=True, env=child_env
    )
    # Extract just the two ``CREATE OR REPLACE TABLE``
    # statements so the cell's output isn't dominated by
    # the property-graph DDL we already showed in Beat 1.
    ci_start = compile_out.find("CREATE OR REPLACE TABLE")
    assert ci_start >= 0, (
        "no CREATE OR REPLACE TABLE in compile output; "
        "concept-index emission may have failed."
    )
    concept_index_ddl = compile_out[ci_start:]
    # Print first ~20 lines so the structure is visible
    # without flooding the notebook.
    preview = concept_index_ddl.splitlines()
    for line in preview[:25]:
        print(line)
    if len(preview) > 25:
        print(f"... ({len(preview) - 25} more lines)")


/usr/local/opt/python@3.13/bin/python3.13 -m bigquery_ontology.cli compile --emit-concept-index --concept-index-table test-project-0728-467323.migration_v5_demo_90a2a04b.mako_concept_index --ontology examples/migration_v5/ontology.yaml examples/migration_v5/binding.yaml


CREATE OR REPLACE TABLE `test-project-0728-467323.migration_v5_demo_90a2a04b.mako_concept_index`
AS SELECT * FROM UNNEST(ARRAY<STRUCT<entity_name STRING, label STRING, label_kind STRING, notation STRING, scheme STRING, language STRING, is_abstract BOOL, compile_id STRING, compile_fingerprint STRING>>[
  ('AgentSession', 'AgentSession', 'name', NULL, NULL, NULL, FALSE, '0932d713df32', '0932d713df32e202cb439e1d937d41d6ac6a958f75406149ebdf905b31078787'),
  ('BusinessConstraint', 'BusinessConstraint', 'name', NULL, NULL, NULL, FALSE, '0932d713df32', '0932d713df32e202cb439e1d937d41d6ac6a958f75406149ebdf905b31078787'),
  ('Candidate', 'Candidate', 'name', NULL, NULL, NULL, FALSE, '0932d713df32', '0932d713df32e202cb439e1d937d41d6ac6a958f75406149ebdf905b31078787'),
  ('ConstraintApplication', 'ConstraintApplication', 'name', NULL, NULL, NULL, FALSE, '0932d713df32', '0932d713df32e202cb439e1d937d41d6ac6a958f75406149ebdf905b31078787'),
  ('ContextSnapshot', 'ContextSnapshot', 'name', NULL, NULL, 

### 4.3 — apply the index + resolve a user-typed label

Apply the `CREATE OR REPLACE TABLE` statements to the scratch dataset, then
construct `OntologyRuntime` with `concept_index_table` set. `LabelSynonymResolver`
wraps the runtime's `concept_index` lookup; `resolver.resolve(query)` returns
ranked `ResolverCandidate` rows.

The candidate carries:
- `entity_name` — canonical entity
- `matched_label` / `matched_label_kind` — what the index matched against
  (`name` / `pref` / `alt` / `hidden` / `synonym` / `notation`)
- `compile_id` / `compile_fingerprint` — provenance: which compile produced
  the index this row came from

In [24]:
if not FEATURES["concept_index_reader"]:
    print("Skipped: concept_index_reader feature off")
else:
    # Apply the two CREATE OR REPLACE TABLE statements.
    # The compile output may contain multiple statements
    # separated by ``;\n``; running them one at a time keeps
    # the queries small and the error messages clear.
    for stmt in concept_index_ddl.split(";"):
        stmt = stmt.strip()
        if not stmt:
            continue
        bq.query(stmt + ";").result()
    print(f"Concept index materialized to {CONCEPT_INDEX_TABLE}")

    # Read the compiler version straight from the __meta
    # row the compiler just emitted. Hard-coding a literal
    # (``"bigquery_ontology 0.2.2"``) would drift the moment
    # the package version bumps: the compile_fingerprint
    # closes over the version string, so any mismatch
    # between what was emitted and what the runtime is told
    # trips FingerprintMismatchError. The __meta row is the
    # SDK's own authoritative answer.
    meta_row = next(iter(bq.query(
        f"SELECT compiler_version, compile_fingerprint "
        f"FROM `{CONCEPT_INDEX_TABLE}__meta`"
    ).result()))
    COMPILER_VERSION = meta_row.compiler_version
    print(f"compiler_version    = {COMPILER_VERSION!r}")
    print(f"compile_fingerprint = {meta_row.compile_fingerprint}")

    from bigquery_agent_analytics.ontology_runtime import (
        OntologyRuntime, LabelSynonymResolver,
    )
    runtime = OntologyRuntime.from_files(
        ontology_path=str(ONTOLOGY_PATH),
        binding_path=str(BINDING_PATH),
        compiler_version=COMPILER_VERSION,
        concept_index_table=CONCEPT_INDEX_TABLE,
        bq_client=bq,
    )
    resolver = LabelSynonymResolver(runtime)

    # Resolve a user-typed query against the canonical
    # entity names. With a richer TTL, the same call
    # resolves ``"audience segment"`` → ``Candidate``
    # (via skos:altLabel) — only the underlying ontology
    # changes, the resolver call doesn't.
    user_query = "DecisionExecution"
    candidates = resolver.resolve(user_query, limit=5)
    print(f"resolver.resolve({user_query!r}) → {len(candidates)} candidate(s)")
    for c in candidates:
        print(
            f"  entity={c.entity_name:24s} "
            f"label={c.matched_label!r} kind={c.matched_label_kind} "
            f"compile_id={c.compile_id}"
        )
    assert candidates, (
        f"expected at least one candidate for {user_query!r}; "
        f"concept-index may not have been applied."
    )
    # Lock the canonical entity for cell 4.4's GQL.
    resolved_entity = candidates[0].entity_name
    resolved_compile_id = candidates[0].compile_id


Concept index materialized to test-project-0728-467323.migration_v5_demo_90a2a04b.mako_concept_index


compiler_version    = 'bigquery_ontology 0.2.2'
compile_fingerprint = 0932d713df32e202cb439e1d937d41d6ac6a958f75406149ebdf905b31078787


resolver.resolve('DecisionExecution') → 1 candidate(s)
  entity=DecisionExecution        label='DecisionExecution' kind=name compile_id=0932d713df32


### 4.4 — GQL traversal scoped by the resolved entity

Once `resolver.resolve(user_query)` returns a canonical `entity_name`, the GQL
traversal plugs it in as the `MATCH` label. Below: count `DecisionExecution`
nodes for the sessions Section 0 populated, then run the hub-shape sample
(`(DecisionExecution)-[partOfSession]->(AgentSession)`) — now non-zero because
the **compiled extractor** wired by Beat 3.5 synthesizes `AgentSession` +
`partOfSession` from the plugin envelope for every `complete_execution` event.
(The reference extractor is the fallback if compiled output ever fails
validation; in this run it's bypassed.)

This is the climax query: user-typed label (canonical in this fixture; natural-language `skos:altLabel` / `skos:prefLabel` rows in richer ontologies) → resolver returns a canonical entity → traversal works against the property graph the user authored.

In [25]:
if not FEATURES["concept_index_reader"]:
    print("Skipped: concept_index_reader feature off")
else:
    # The resolved entity is a string, but GoogleSQL's
    # ``GRAPH_TABLE`` MATCH label is a static identifier,
    # not a parameter — so build the SQL via f-string after
    # asserting the resolved entity is in the demo allowlist.
    # The check is defence-in-depth: ``LabelSynonymResolver``
    # only returns rows from the SDK-emitted concept index,
    # which is bounded by the ontology, but a future
    # multi-ontology setup might need stricter scoping.
    assert resolved_entity in {
        "AgentSession", "Candidate", "ContextSnapshot",
        "DecisionExecution", "DecisionPoint", "SelectionOutcome",
    }, f"resolved entity {resolved_entity!r} outside demo allowlist"

    # Entity → PK column map (matches binding.yaml). The GQL
    # MATCH label is a static identifier, but the COLUMNS
    # accessor needs the entity-specific PK column name —
    # ``id`` no longer exists on the underlying table since
    # ``make_binding`` renamed the PK column per-entity.
    _PK_COL = {
        "AgentSession": "agent_session_id",
        "Candidate": "candidate_id",
        "ContextSnapshot": "context_snapshot_id",
        "DecisionExecution": "decision_execution_id",
        "DecisionPoint": "decision_point_id",
        "SelectionOutcome": "selection_outcome_id",
    }
    pk_col = _PK_COL[resolved_entity]

    # Scope to the sessions Section 0 populated. Stale rows
    # from earlier runs (or AI-hallucinated PK values without
    # a session prefix) would otherwise inflate the count and
    # mask a regression where the compiled extractor stopped
    # producing this run's envelope-side ``AgentSession`` +
    # ``partOfSession``. The compiled extractor's PK values
    # always start with ``{session_id}:``; filter on that.
    import re
    # The session-prefix regex below assumes the resolved
    # entity's PK column is ``{session_id}:{raw_id}``. That's
    # true for every demo entity *except* ``AgentSession``,
    # whose ``agent_session_id`` is exactly the session_id
    # with no colon. The narrative counts ``DecisionExecution``;
    # the assert keeps the contract explicit so a future edit
    # that swaps the resolved entity won't silently emit a
    # regex that matches nothing.
    assert resolved_entity == "DecisionExecution", (
        f"scoped GQL below assumes per-session PK prefixes; "
        f"the demo's resolved entity must be DecisionExecution "
        f"(got {resolved_entity!r})."
    )
    _session_re = "^(" + "|".join(re.escape(sid) for sid in session_ids) + "):"
    count_gql = f"""
        SELECT COUNT(*) AS n
        FROM GRAPH_TABLE(
          `{PROJECT_ID}.{DATASET_ID}.mako_demo_graph`
          MATCH (n:{resolved_entity})
          WHERE REGEXP_CONTAINS(n.{pk_col}, @session_re)
          COLUMNS (n.{pk_col} AS id)
        )
    """
    count = next(iter(bq.query(
        count_gql,
        job_config=bigquery.QueryJobConfig(query_parameters=[
            bigquery.ScalarQueryParameter(
                "session_re", "STRING", _session_re
            ),
        ]),
    ).result())).n
    print(
        f"GRAPH_TABLE COUNT on {resolved_entity} "
        f"(compile_id={resolved_compile_id}) = {count}"
    )
    # The resolver-to-graph wiring is only meaningful if the
    # resolved entity has rows in the base tables. A zero
    # count means either Section 0 populated agent_events
    # but Beat 1 didn't materialize rows for this entity, or
    # the resolver returned an entity we materialize zero of
    # — either way the demo's climax claim ("user-typed
    # name routes to real graph data") would be empty.
    assert count > 0, (
        f"GRAPH_TABLE matched zero {resolved_entity} nodes; "
        f"the build in Beat 1 may have produced an empty graph."
    )

    # Hub-shape traversal: for each DecisionExecution
    # node, follow ``partOfSession`` to the AgentSession
    # it belongs to. Beat 3.5's **compiled extractor**
    # synthesizes ``AgentSession`` + ``partOfSession``
    # from the plugin envelope (the reference extractor
    # would do the same as fallback, but compiled output
    # is validator-clean so fallback is bypassed). The
    # hub-shape MATCH should now return >= 1 row.
    if resolved_entity == "DecisionExecution":
        # Same session-scope filter for the hub-shape sample.
        hub_gql = f"""
            SELECT
              de_id, s_id
            FROM GRAPH_TABLE(
              `{PROJECT_ID}.{DATASET_ID}.mako_demo_graph`
              MATCH (de:DecisionExecution)-[:partOfSession]->(s:AgentSession)
              WHERE REGEXP_CONTAINS(de.decision_execution_id, @session_re)
              COLUMNS (de.decision_execution_id AS de_id, s.agent_session_id AS s_id)
            )
            LIMIT 50
        """
        hub_rows = list(bq.query(
            hub_gql,
            job_config=bigquery.QueryJobConfig(query_parameters=[
                bigquery.ScalarQueryParameter(
                    "session_re", "STRING", _session_re
                ),
            ]),
        ).result())
        # Hub-shape MUST return rows for every current session.
        # Beat 3.5's compiled extractor synthesizes
        # ``AgentSession`` + ``partOfSession`` for every
        # ``complete_execution`` event. A zero result here would
        # mean the compiled bundle stopped producing the
        # envelope-side synthesis — fail hard so a regression
        # doesn't slip past with a "no data" print.
        assert hub_rows, (
            "hub-shape MATCH returned zero rows for the current "
            "sessions — Beat 3.5's compiled extractor should have "
            "synthesized AgentSession + partOfSession for every "
            "complete_execution."
        )
        observed_sessions = {row.s_id for row in hub_rows}
        missing_sessions = set(session_ids) - observed_sessions
        assert not missing_sessions, (
            f"hub-shape missing sessions: {missing_sessions}; "
            f"observed: {observed_sessions}"
        )
        print()
        print(f"{'decision_execution_id':38s} {'agent_session_id':38s}")
        print("-" * 78)
        for row in hub_rows:
            print(f"{row.de_id:38s} {row.s_id:38s}")


GRAPH_TABLE COUNT on DecisionExecution (compile_id=0932d713df32) = 9



decision_execution_id                  agent_session_id                      
------------------------------------------------------------------------------
4bb27dc9-dcef-49d1-b21a-fb530b2811ef:exec-3d9f6e0f48 4bb27dc9-dcef-49d1-b21a-fb530b2811ef  
4bb27dc9-dcef-49d1-b21a-fb530b2811ef:exec-3d9f6e0f48 4bb27dc9-dcef-49d1-b21a-fb530b2811ef  
4bb27dc9-dcef-49d1-b21a-fb530b2811ef:exec-3d9f6e0f48 4bb27dc9-dcef-49d1-b21a-fb530b2811ef  
4bb27dc9-dcef-49d1-b21a-fb530b2811ef:exec-3d9f6e0f48 4bb27dc9-dcef-49d1-b21a-fb530b2811ef  
4bb27dc9-dcef-49d1-b21a-fb530b2811ef:exec-3d9f6e0f48 4bb27dc9-dcef-49d1-b21a-fb530b2811ef  
4bb27dc9-dcef-49d1-b21a-fb530b2811ef:exec-3d9f6e0f48 4bb27dc9-dcef-49d1-b21a-fb530b2811ef  
4bb27dc9-dcef-49d1-b21a-fb530b2811ef:exec-3d9f6e0f48 4bb27dc9-dcef-49d1-b21a-fb530b2811ef  
4bb27dc9-dcef-49d1-b21a-fb530b2811ef:exec-3d9f6e0f48 4bb27dc9-dcef-49d1-b21a-fb530b2811ef  
4bb27dc9-dcef-49d1-b21a-fb530b2811ef:exec-3d9f6e0f48 4bb27dc9-dcef-49d1-b21a-fb530b2811ef  
624b3128-7bb0-

Beat 4 is closed:

- The SDK emitted a concept index alongside the property graph at compile time
  (cell 4.2). Each row carries `compile_id` + `compile_fingerprint` for
  provenance.
- The user-typed query resolved to a canonical entity via `LabelSynonymResolver`
  (cell 4.3). The resolver returns the same `compile_fingerprint` it loaded the
  index with — drift between the compile that produced the graph and the compile
  that produced the index would surface as a mismatched fingerprint.
- The resolved canonical name plugs straight into a GQL traversal against the
  user-authored property graph (cell 4.4).
- The hub-shape variant of that traversal (`DecisionExecution -[partOfSession]->
  AgentSession`) returns non-zero rows: the **compiled extractor** wired in Beat
  3.5 (handling `complete_execution` events) synthesizes the envelope-side
  `AgentSession` node + `partOfSession` edge. The reference extractor is the
  fallback if the compiled output ever fails validation; in this run the
  compiled path covers every `complete_execution` cleanly so fallback is
  bypassed.

In this fixture the user typed the canonical entity label (`DecisionExecution`)
because the MAKO TTL ships no SKOS labels — the concept index has one row per
entity with `label_kind='name'`. The same resolver call —
`LabelSynonymResolver.resolve(query)` — handles richer ontologies that emit
`skos:altLabel` / `skos:prefLabel` / `skos:notation` rows: the resolver picks up
the additional rows automatically and re-ranks by label kind
(`name` > `pref` > `alt` > `hidden` > `synonym` > `notation`). Only the ontology changes;
the notebook code doesn't.

## Section 5 — Beat 5: feedback / reward loop closes the audit story (#187)

**Guarantee:** Close the loop. Beats 1–4 prove the decision-time graph is
owned, validated, extracted cheaply, and addressable by canonical name.
Beat 5 proves the *after-the-decision* half: observed real-world outcomes
flow back into the same graph, every losing candidate gets a recorded
reason, and an RL training signal can be derived from those outcomes —
all through the same SDK extraction + materialization path the prior beats use.

The MAKO agent now emits four additional `TOOL_COMPLETED` events per
decision (see `examples/migration_v5/mako_demo_agent.py:apply_constraint`,
`record_rejection`, `record_outcome_signal`, `compute_reward`). The
reference extractor at `examples/migration_v5/reference_extractor.py`
turns each into Beat 5 nodes + edges; the materializer writes them to
the same BigQuery dataset the Beats 1–4 hub lives in. The cells below
extract, materialize, and run the two payoff GQL traversals against
the live property graph.

### 5.1 — extract Beat 5 events with the reference extractor

The standard `bqaa ontology-build` discovery picks up the same sessions
Beat 1 found, but the AI fallback alone won't reliably structure the
Beat 5 tool events (the agent's free-text `signal_type` /
`rejection_text` payloads aren't in MAKO's declared property set).
The reference extractor at `examples/migration_v5/reference_extractor.py`
covers each Beat 5 tool deterministically — wire it onto
`OntologyGraphManager` via the `extractors=` kwarg and re-extract.

In [26]:
if not FEATURES.get("beat5_feedback_loop", True):
    print("Skipped: beat5_feedback_loop feature off")
else:
    import sys as _sys
    _sys.path.insert(0, "examples/migration_v5")
    import reference_extractor
    from bigquery_agent_analytics.ontology_graph import OntologyGraphManager

    mgr_beat5 = OntologyGraphManager.from_ontology_binding(
        project_id=PROJECT_ID,
        dataset_id=DATASET_ID,
        ontology=ontology_obj,
        binding=binding_obj,
        bq_client=bq,
        extractors=reference_extractor.EXTRACTORS,
    )
    graph_beat5 = mgr_beat5.extract_graph(
        session_ids=session_ids,
        use_ai_generate=False,
        run_structured=True,
        on_unhandled_span="stub",
    )
    by_entity = {}
    for n in graph_beat5.nodes:
        by_entity[n.entity_name] = by_entity.get(n.entity_name, 0) + 1
    by_rel = {}
    for e in graph_beat5.edges:
        by_rel[e.relationship_name] = by_rel.get(e.relationship_name, 0) + 1
    print(f"extracted nodes={len(graph_beat5.nodes)} edges={len(graph_beat5.edges)}")
    print("Beat 5 nodes:", {k: v for k, v in by_entity.items()
                            if k in ("OutcomeSignal", "RewardComputation",
                                     "RejectionReason", "BusinessConstraint",
                                     "ConstraintApplication")})
    print("Beat 5 edges:", {k: v for k, v in by_rel.items()
                            if k in ("producedOutcome", "derivedReward",
                                     "hasRejectionReason",
                                     "appliedConstraint",
                                     "filteredByConstraint")})
    assert any(n.entity_name == "OutcomeSignal" for n in graph_beat5.nodes), \
        "no OutcomeSignal nodes extracted — re-run Section 0 against an agent build that emits Beat 5 tools"
    assert any(n.entity_name == "RewardComputation" for n in graph_beat5.nodes), \
        "no RewardComputation nodes extracted"
    assert any(n.entity_name == "RejectionReason" for n in graph_beat5.nodes), \
        "no RejectionReason nodes extracted"


User-provided bigquery.Client is not a LabeledBigQueryClient; SDK telemetry labels will not be applied to jobs from this client. To opt in, construct the client via bigquery_agent_analytics.make_bq_client() or pass a LabeledBigQueryClient directly.


extracted nodes=113 edges=42
Beat 5 nodes: {'RejectionReason': 6, 'OutcomeSignal': 6, 'RewardComputation': 3}
Beat 5 edges: {'hasRejectionReason': 6, 'producedOutcome': 6, 'derivedReward': 6}


### 5.2 — materialize the Beat 5 rows

`OntologyMaterializer.materialize_with_status` writes the extracted nodes
+ edges to the binding's BigQuery tables. The new Beat 5 tables
(`outcome_signal`, `reward_computation`, `rejection_reason`,
`business_constraint`, `constraint_application` + their edge tables) were
created by Section 0's `table_ddl.sql` apply — `material­ize_with_status`
just INSERTs into them, same path Beat 1–4 entities take.

In [27]:
if not FEATURES.get("beat5_feedback_loop", True):
    print("Skipped: beat5_feedback_loop feature off")
else:
    from bigquery_agent_analytics.ontology_materializer import OntologyMaterializer

    mat_beat5 = OntologyMaterializer(
        spec=mgr_beat5.spec,
        project_id=PROJECT_ID,
        dataset_id=DATASET_ID,
        location=DATASET_LOCATION,
        bq_client=bq,
    )
    mat_result = mat_beat5.materialize_with_status(
        graph_beat5, session_ids=session_ids
    )
    beat5_counts = {
        k: v for k, v in mat_result.row_counts.items()
        if k in ("OutcomeSignal", "RewardComputation", "RejectionReason",
                 "BusinessConstraint", "ConstraintApplication",
                 "producedOutcome", "derivedReward", "hasRejectionReason",
                 "appliedConstraint", "filteredByConstraint")
    }
    print(f"Beat 5 row_counts: {beat5_counts}")
    # Every Beat 5 entity that came out of extraction must land in BQ —
    # a row-count of zero on the materializer side after non-zero
    # extraction would mean the binding rejected the row at INSERT time.
    for kind, count in beat5_counts.items():
        assert count > 0, f"materializer wrote zero rows for {kind!r}"


User-provided bigquery.Client is not a LabeledBigQueryClient; SDK telemetry labels will not be applied to jobs from this client. To opt in, construct the client via bigquery_agent_analytics.make_bq_client() or pass a LabeledBigQueryClient directly.


Delete for sessions failed on test-project-0728-467323.migration_v5_demo_90a2a04b.context_snapshot: 400 GET https://bigquery.googleapis.com/bigquery/v2/projects/test-project-0728-467323/queries/1bd5ae27-d65f-4570-a865-441ab8dff268?maxResults=0&location=US&prettyPrint=false: UPDATE or DELETE statement over table test-project-0728-467323.migration_v5_demo_90a2a04b.context_snapshot would affect rows in the streaming buffer, which is not supported

Location: US
Job ID: 1bd5ae27-d65f-4570-a865-441ab8dff268



Delete for sessions failed on test-project-0728-467323.migration_v5_demo_90a2a04b.decision_point: 400 GET https://bigquery.googleapis.com/bigquery/v2/projects/test-project-0728-467323/queries/cb67fe57-e493-4657-8412-ac72b7eb9eca?maxResults=0&location=US&prettyPrint=false: UPDATE or DELETE statement over table test-project-0728-467323.migration_v5_demo_90a2a04b.decision_point would affect rows in the streaming buffer, which is not supported

Location: US
Job ID: cb67fe57-e493-4657-8412-ac72b7eb9eca



Delete for sessions failed on test-project-0728-467323.migration_v5_demo_90a2a04b.candidate: 400 GET https://bigquery.googleapis.com/bigquery/v2/projects/test-project-0728-467323/queries/73cdc66f-b3da-4049-b035-c35013721cb3?maxResults=0&location=US&prettyPrint=false: UPDATE or DELETE statement over table test-project-0728-467323.migration_v5_demo_90a2a04b.candidate would affect rows in the streaming buffer, which is not supported

Location: US
Job ID: 73cdc66f-b3da-4049-b035-c35013721cb3



Delete for sessions failed on test-project-0728-467323.migration_v5_demo_90a2a04b.selection_outcome: 400 GET https://bigquery.googleapis.com/bigquery/v2/projects/test-project-0728-467323/queries/ec3076e1-de10-4d4e-a88e-de5a9e93aebd?maxResults=0&location=US&prettyPrint=false: UPDATE or DELETE statement over table test-project-0728-467323.migration_v5_demo_90a2a04b.selection_outcome would affect rows in the streaming buffer, which is not supported

Location: US
Job ID: ec3076e1-de10-4d4e-a88e-de5a9e93aebd



...

Delete for sessions failed on test-project-0728-467323.migration_v5_demo_90a2a04b.part_of_session: 400 GET https://bigquery.googleapis.com/bigquery/v2/projects/test-project-0728-467323/queries/97d58228-7878-49e4-84b0-7348d4cd8b1f?maxResults=0&location=US&prettyPrint=false: UPDATE or DELETE statement over table test-project-0728-467323.migration_v5_demo_90a2a04b.part_of_session would affect rows in the streaming buffer, which is not supported

Location: US
Job ID: 97d58228-7878-49e4-84b0-7348d4cd8b1f



Delete for sessions failed on test-project-0728-467323.migration_v5_demo_90a2a04b.has_rejection_reason: 400 GET https://bigquery.googleapis.com/bigquery/v2/projects/test-project-0728-467323/queries/ee34b5b6-f9be-4da9-a074-ebf39c52ddc9?maxResults=0&location=US&prettyPrint=false: UPDATE or DELETE statement over table test-project-0728-467323.migration_v5_demo_90a2a04b.has_rejection_reason would affect rows in the streaming buffer, which is not supported

Location: US
Job ID: ee34b5b6-f9be-4da9-a074-ebf39c52ddc9



Delete for sessions failed on test-project-0728-467323.migration_v5_demo_90a2a04b.produced_outcome: 400 GET https://bigquery.googleapis.com/bigquery/v2/projects/test-project-0728-467323/queries/50b84ded-c3b4-4bcf-b826-82cba0b73267?maxResults=0&location=US&prettyPrint=false: UPDATE or DELETE statement over table test-project-0728-467323.migration_v5_demo_90a2a04b.produced_outcome would affect rows in the streaming buffer, which is not supported

Location: US
Job ID: 50b84ded-c3b4-4bcf-b826-82cba0b73267



Delete for sessions failed on test-project-0728-467323.migration_v5_demo_90a2a04b.derived_reward: 400 GET https://bigquery.googleapis.com/bigquery/v2/projects/test-project-0728-467323/queries/fc3c807e-4b97-4fab-bf22-8cc21a3428e8?maxResults=0&location=US&prettyPrint=false: UPDATE or DELETE statement over table test-project-0728-467323.migration_v5_demo_90a2a04b.derived_reward would affect rows in the streaming buffer, which is not supported

Location: US
Job ID: fc3c807e-4b97-4fab-bf22-8cc21a3428e8



Beat 5 row_counts: {'RejectionReason': 6, 'OutcomeSignal': 6, 'RewardComputation': 3, 'hasRejectionReason': 6, 'producedOutcome': 6, 'derivedReward': 6}


### 5.3 — payoff traversal #1: which signals fed which reward?

The reward-loop GQL: walk from each `DecisionExecution` through
`producedOutcome` to the `OutcomeSignal`s it generated, then backwards
through `derivedReward` to the `RewardComputation` that aggregated those
signals into a scalar. This is the trace an RL pipeline reads when it
needs to attribute training data back to specific decisions.

In [28]:
if not FEATURES.get("beat5_feedback_loop", True):
    print("Skipped: beat5_feedback_loop feature off")
else:
    reward_gql = f"""
        SELECT de_id, signal_id, reward_id, reward_value
        FROM GRAPH_TABLE(
          `{PROJECT_ID}.{DATASET_ID}.mako_demo_graph`
          MATCH (de:DecisionExecution)-[:producedOutcome]->(s:OutcomeSignal)
                <-[:derivedReward]-(rc:RewardComputation)
          COLUMNS (de.decision_execution_id AS de_id,
                   s.outcome_signal_id AS signal_id,
                   rc.reward_computation_id AS reward_id,
                   rc.reward_value AS reward_value)
        )
        ORDER BY de_id, signal_id
    """
    reward_rows = list(bq.query(reward_gql).result())
    assert reward_rows, (
        "reward-loop GQL returned zero rows — Beat 5 materialization "
        "didn't wire DecisionExecution -> producedOutcome -> OutcomeSignal "
        "<- derivedReward <- RewardComputation"
    )
    print(f"{'de_id (last 12)':14s} {'signal (last 12)':18s} "
          f"{'reward (last 12)':18s} {'reward_value':>13s}")
    print("-" * 70)
    for row in reward_rows:
        print(f"{row.de_id[-12:]:14s} {row.signal_id[-12:]:18s} "
              f"{row.reward_id[-12:]:18s} {row.reward_value:>13.3f}")


de_id (last 12) signal (last 12)   reward (last 12)    reward_value
----------------------------------------------------------------------
c-3d9f6e0f48   g-0549532911       w-576d438bdd               0.900
c-3d9f6e0f48   g-0549532911       w-576d438bdd               0.900
c-3d9f6e0f48   g-0549532911       w-576d438bdd               0.900
c-3d9f6e0f48   g-0549532911       w-576d438bdd               0.900
c-3d9f6e0f48   g-0549532911       w-576d438bdd               0.900
c-3d9f6e0f48   g-0549532911       w-576d438bdd               0.900
c-3d9f6e0f48   g-0549532911       w-576d438bdd               0.900
c-3d9f6e0f48   g-0549532911       w-576d438bdd               0.900
c-3d9f6e0f48   g-0549532911       w-576d438bdd               0.900
c-3d9f6e0f48   g-0549532911       w-576d438bdd               0.900
c-3d9f6e0f48   g-0549532911       w-576d438bdd               0.900
c-3d9f6e0f48   g-0549532911       w-576d438bdd               0.900
c-3d9f6e0f48   g-0549532911       w-576d438bdd           

### 5.4 — payoff traversal #2: why did each candidate lose?

The audit GQL: walk from each `Candidate` through `hasRejectionReason`
to its `RejectionReason`. Operators reading the graph after a decision
can answer "show me every candidate that lost and why" in one MATCH.
The `selectedCandidate` edge (already in the graph from Beat 4) covers
the winning candidate; `hasRejectionReason` covers the rest.

In [29]:
if not FEATURES.get("beat5_feedback_loop", True):
    print("Skipped: beat5_feedback_loop feature off")
else:
    rejection_gql = f"""
        SELECT cand_id, rej_id
        FROM GRAPH_TABLE(
          `{PROJECT_ID}.{DATASET_ID}.mako_demo_graph`
          MATCH (c:Candidate)-[:hasRejectionReason]->(r:RejectionReason)
          COLUMNS (c.candidate_id AS cand_id,
                   r.rejection_reason_id AS rej_id)
        )
        ORDER BY cand_id
    """
    rejection_rows = list(bq.query(rejection_gql).result())
    assert rejection_rows, (
        "rejection GQL returned zero rows — Beat 5 should record at least "
        "one hasRejectionReason edge per losing candidate"
    )
    print(f"{'candidate (last 12)':22s} {'rejection (last 12)':22s}")
    print("-" * 46)
    for row in rejection_rows:
        print(f"{row.cand_id[-12:]:22s} {row.rej_id[-12:]:22s}")


candidate (last 12)    rejection (last 12)   
----------------------------------------------
d-2d0ff00858           j-2ca024cdd0          
d-2d0ff00858           j-2ca024cdd0          
d-2d0ff00858           j-2ca024cdd0          
d-2d0ff00858           j-2ca024cdd0          
d-2d0ff00858           j-2ca024cdd0          
d-2d0ff00858           j-2ca024cdd0          
d-2d0ff00858           j-2ca024cdd0          
d-2d0ff00858           j-2ca024cdd0          
d-2d0ff00858           j-2ca024cdd0          
d-2d0ff00858           j-2ca024cdd0          
d-2d0ff00858           j-2ca024cdd0          
d-2d0ff00858           j-2ca024cdd0          
d-2d0ff00858           j-2ca024cdd0          
d-2d0ff00858           j-2ca024cdd0          
d-2d0ff00858           j-2ca024cdd0          
d-2d0ff00858           j-2ca024cdd0          
d-2d0ff00858           j-2ca024cdd0          
d-2d0ff00858           j-2ca024cdd0          
d-2d0ff00858           j-2ca024cdd0          
d-2d0ff00858           j-2ca024cd

Beat 5 is closed:

- The reference extractor produced Beat 5 nodes (`OutcomeSignal`,
  `RewardComputation`, `RejectionReason`) from the agent's
  `TOOL_COMPLETED` events (cell 5.1).
- The materializer wrote them — plus the connecting edges
  (`producedOutcome`, `derivedReward`, `hasRejectionReason`) — to the
  Beat 5 BigQuery tables created by Section 0's DDL (cell 5.2).
- The reward-loop traversal (`DecisionExecution -> producedOutcome ->
  OutcomeSignal <- derivedReward <- RewardComputation`, cell 5.3) returns
  non-zero rows with the actual `reward_value` per execution — the trace
  an RL training pipeline reads.
- The rejection traversal (`Candidate -> hasRejectionReason ->
  RejectionReason`, cell 5.4) returns non-zero rows — every losing
  candidate is attributable to a recorded reason.

The decision-and-feedback loop now closes end-to-end inside the same
graph the prior four beats populated.

## Section 6 — close

### Four-guarantee recap

| Guarantee | Before | After |
|---|---|---|
| **Own** | `CREATE OR REPLACE PROPERTY GRAPH` every build (SDK overwrites your DDL) | `--skip-property-graph`; you own the graph object, the SDK populates base tables. |
| **Validate** | First failure is at extraction time, after `AI.GENERATE` already ran | Sub-second pre-flight against live BigQuery schema before extraction starts. |
| **Extract cheaply** | `AI.GENERATE` over the full transcript every session | Compiled deterministic extractors handle structured events; AI fallback only for semantic gaps. Cell 3.2 establishes the pre-compile baseline; cell 3.7 shows the compiled-runtime transcript-size delta (live). |
| **Resolve** | User types an entity label and the GQL query depends on knowing its canonical name | SKOS concept-index lookup resolves user-typed inputs to canonical names before GQL runs. In this fixture the input is the canonical entity name; richer ontologies that ship `skos:altLabel` / `skos:prefLabel` rows resolve natural-language inputs via the same call. |

### Read more

Each guarantee has its own issue + landing PRs:

- **Own** — [#104](https://github.com/GoogleCloudPlatform/BigQuery-Agent-Analytics-SDK/issues/104)
- **Validate** — [#105](https://github.com/GoogleCloudPlatform/BigQuery-Agent-Analytics-SDK/issues/105) (pre-flight) + [#76](https://github.com/GoogleCloudPlatform/BigQuery-Agent-Analytics-SDK/issues/76) (post-extract `ValidationReport`)
- **Extract cheaply** — [#75](https://github.com/GoogleCloudPlatform/BigQuery-Agent-Analytics-SDK/issues/75) (compile-extractors C1 + C2)
- **Resolve** — [#58](https://github.com/GoogleCloudPlatform/BigQuery-Agent-Analytics-SDK/issues/58) (concept-index reader; emission shipped in [PR #92](https://github.com/GoogleCloudPlatform/BigQuery-Agent-Analytics-SDK/pull/92))
- **Storyboard** — [#107](https://github.com/GoogleCloudPlatform/BigQuery-Agent-Analytics-SDK/issues/107) (this notebook's cell-by-cell plan)

### What's next

- A default `@builtin:adk-events` ontology so users with no domain-specific extraction needs can run the SDK with **zero authored YAML**.
- Phase 2 of [#75](https://github.com/GoogleCloudPlatform/BigQuery-Agent-Analytics-SDK/issues/75): session-aggregated compilation that reduces per-session AI cost further.
- Additional resolver layers in [#58](https://github.com/GoogleCloudPlatform/BigQuery-Agent-Analytics-SDK/issues/58) beyond `LabelSynonymResolver` (semantic similarity, embedding-based).